# QWZ Model Simulation

This notebook works through the Qi-Wu-Zhang 2-band model. The aim is to set up the canonical form of the model and confirm its topology using the standard berry curvature method and spectral localiser (which requires a real-space expression). After this, additional terms can be introduced to alter the bandgap, creating an indirect bandgap. Now, fractional Chern numbers can be calcualted by noth methods. Thereafter, disorder of various kinds can be added and tuned to investigate the effect on topology in the direct and indirect bandgap limits.



In [ ]:
using LinearAlgebra
using Printf
using Plots
using DataFrames
using SparseArrays
using ProgressMeter
using Statistics
using LaTeXStrings
using JLD2: @save, @load

using KrylovKit
using LDLFactorizations

In [ ]:
## Pauli matrices
sigma_x = [0 1; 1 0]
sigma_y = [0 -im; im 0]
sigma_z = [1 0; 0 -1]
identity = [1 0; 0 1]

## Hamiltonian (k-space)

Let:
$$\vec\sigma = (\sigma_x, \sigma_y, \sigma_z)$$
$$\vec d(\mathbf{k}) = (d_x(\mathbf{k}), d_y(\mathbf{k}), d_z(\mathbf{k}))$$

where
$$d_x(\mathbf{k}) = A\sin{(k_x})$$
$$d_y(\mathbf{k}) = A\sin{(k_y})$$
$$d_z(\mathbf{k}) = m + B(2 - \cos{(k_x)}-\cos{(k_y)})$$
\begin{equation}
\begin{split}
&\text{set} \ B=1 \\
&= (m+2) - \cos{(k_x)}-\cos{(k_y)} \\
&= M - \cos{(k_x)}-\cos{(k_y)}
\end{split}
\end{equation}

The Hamiltonian is
$$H(\mathbf{k}) = d_x(\mathbf{k})\sigma_x + d_y(\mathbf{k})\sigma_y + d_z(\mathbf{k})\sigma_z$$

Which can be solved by taking $H(\mathbf{k})^2=|\vec d|^2\mathbf{I}$ to give (N.B. taking $A=B=1$)
$$E_\pm(\mathbf{k}) = \pm|\vec d(\mathbf{k})| = \pm\sqrt{\sin^2{(k_x)} + \sin^2{(k_y)} + (M-\cos(k_x) - \cos{(k_y)})^2}$$


In [3]:
println(collect(range(1e-3, 5e-2, 15)))

[0.001, 0.0045, 0.008, 0.0115, 0.015, 0.0185, 0.022, 0.0255, 0.029, 0.0325, 0.036, 0.0395, 0.043, 0.0465, 0.05]


In [ ]:
# QWZ Hamiltonian with general A, B, m
function hamiltonian_qwz(
    kx::Real,
    ky::Real;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0
    )::Matrix{ComplexF64}

    dx::Float64 = Float64(A) * sin(Float64(kx))
    dy::Float64 = Float64(A) * sin(Float64(ky))
    dz::Float64 = Float64(m) + Float64(B) * (2.0 - cos(Float64(kx)) - cos(Float64(ky)))
    return ComplexF64.(dx * sigma_x + dy * sigma_y + dz * sigma_z)
end

function dvec_qwz(
    kx::Real,
    ky::Real;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0
    )::NTuple{3, Float64}
    
    dx::Float64 = Float64(A) * sin(Float64(kx))
    dy::Float64 = Float64(A) * sin(Float64(ky))
    dz::Float64 = Float64(m) + Float64(B) * (2.0 - cos(Float64(kx)) - cos(Float64(ky)))
    return (dx, dy, dz)
end

# Analytic QWZ band energies E_±(k) = ±sqrt(dx^2 + dy^2 + dz^2)
function band_energies_qwz(
    kx::Real,
    ky::Real;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0
    )::NTuple{2, Float64}

    dx::Float64 = Float64(A) * sin(Float64(kx))
    dy::Float64 = Float64(A) * sin(Float64(ky))
    dz::Float64 = Float64(m) + Float64(B) * (2.0 - cos(Float64(kx)) - cos(Float64(ky)))
    e::Float64 = sqrt(dx * dx + dy * dy + dz * dz)
    return (-e, e)
end

# Analytic QWZ band structure over a grid of kx, ky values
function band_structure_qwz(
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0;
    nk::Int=51
    )::Tuple{Vector{Float64}, Vector{Float64}, Array{Float64, 3}}

    ks::Vector{Float64} = collect(range(-pi, pi; length=nk + 1))[1:end-1]  # avoid duplicate endpoint at +pi
    Evals::Array{Float64, 3} = zeros(Float64, length(ks), length(ks), 2)

    for (i, kx) in enumerate(ks), (j, ky) in enumerate(ks)
        e1, e2 = band_energies_qwz(kx, ky; A=A, B=B, m=m)
        Evals[i, j, 1] = e1
        Evals[i, j, 2] = e2
    end

    return ks, ks, Evals
end

# Wide-format table: one row per (A, B, m, kx, ky), with both bands in separate columns.
function build_evals_df(
    Avals::AbstractVector{<:Real},
    Bvals::AbstractVector{<:Real},
    mvals::AbstractVector{<:Real};
    nk::Int=51
    )::DataFrame

    ks::Vector{Float64} = collect(range(-pi, pi; length=nk))[1:end-1]  # avoid duplicate endpoint at +pi
    nrows::Int = length(Avals) * length(Bvals) * length(mvals) * length(ks)^2

    Acol = Vector{Float64}(undef, nrows)
    Bcol = Vector{Float64}(undef, nrows)
    mcol = Vector{Float64}(undef, nrows)
    kxcol = Vector{Float64}(undef, nrows)
    kycol = Vector{Float64}(undef, nrows)
    Eminus_col = Vector{Float64}(undef, nrows)
    Eplus_col = Vector{Float64}(undef, nrows)

    idx::Int = 1
    total_iter = length(Avals) * length(Bvals) * length(mvals) * length(ks)^2
    @showprogress for A in Avals, B in Bvals, m in mvals, kx in ks, ky in ks
        e1, e2 = band_energies_qwz(kx, ky; A=A, B=B, m=m)
        @inbounds begin
            Acol[idx] = Float64(A); Bcol[idx] = Float64(B); mcol[idx] = Float64(m)
            kxcol[idx] = kx; kycol[idx] = ky
            Eminus_col[idx] = e1; Eplus_col[idx] = e2
            idx += 1
        end
    end

    return DataFrame(
        A=Acol,
        B=Bcol,
        m=mcol,
        kx=kxcol,
        ky=kycol,
        E_minus=Eminus_col,
        E_plus=Eplus_col
    )
end

In [ ]:
# Avals = [1.0]#, 2.0, 5.0]
# Bvals = collect(0.0:0.2:2.0) #[0.5, 1.0, 2.0] #
# mvals = collect(-8.0:0.1:2.0) #[-2.0, 0.0, 2.0] #
# nk = 51

# qwz_df = build_evals_df(Avals, Bvals, mvals; nk=nk)
# println("Built DataFrame with $(nrow(qwz_df)) rows and $(ncol(qwz_df)) columns.")

## Plotting Bandstructure

In [ ]:
## plots the qwz bandstructure for the E+ and E- bands separately
function plt_qwz_bandstructure_split(
    df::DataFrame;
    atol::Real=1e-8,
    filename::String="qwz_bandstructure_split.png",
    fixed_variables...
    )
    # Verify required columns exist (using string comparison for robustness)
    required_cols = ["kx", "ky", "E_minus", "E_plus"]
    col_names_str = string.(names(df))
    for c in required_cols
        c in col_names_str || error("Missing required column: $c")
    end

    subdf = df
    for (k, v) in fixed_variables
        string(k) in col_names_str || error("Filter key $(k) is not a DataFrame column.")
        col = subdf[!, k]
        if v isa Real && eltype(col) <: Real
            mask = abs.(Float64.(col) .- Float64(v)) .<= Float64(atol)
            subdf = subdf[mask, :]
        else
            mask = col .== v
            subdf = subdf[mask, :]
        end
    end

    nrow(subdf) > 0 || error("No rows matched the requested fixed variables. Try increasing atol or checking available parameter values.")

    kx_vals = sort(unique(subdf.kx))
    ky_vals = sort(unique(subdf.ky))
    nx = length(kx_vals)
    ny = length(ky_vals)

    ix = Dict(kx_vals[i] => i for i in eachindex(kx_vals))
    iy = Dict(ky_vals[j] => j for j in eachindex(ky_vals))

    Emin = fill(NaN, nx, ny)
    Emax = fill(NaN, nx, ny)

    for row in eachrow(subdf)
        i = ix[row.kx]
        j = iy[row.ky]
        Emin[i, j] = row.E_minus
        Emax[i, j] = row.E_plus
    end

    p1 = surface(
        kx_vals, ky_vals, Emin';
        xlabel="k_x", ylabel="k_y", zlabel="E",
        title="Lower band E_-",
        color=:viridis,
        legend=false
    )
    p2 = surface(
        kx_vals, ky_vals, Emax';
        xlabel="k_x", ylabel="k_y", zlabel="E",
        title="Upper band E_+",
        color=:plasma,
        legend=false
    )

    label_text = isempty(fixed_variables) ? "All rows" : join(["$(k)=$(v)" for (k, v) in fixed_variables], ", ")
    plt = plot(p1, p2; layout=(1, 2), size=(1200, 460), plot_title="QWZ Band Structure " * label_text)
    display(plt)
    savefig(plt, filename)
    # return plt, subdf
end

## plots the qwz bandstructure for the E+ and E- bands together in one plot
function plt_qwz_bandstructure_combined(
    df::DataFrame;
    atol::Real=1e-8,
    filename::String="qwz_bandstructure_combined.png",
    alpha::Real=0.75,
    fixed_variables...
)
    # Verify required columns exist (using string comparison for robustness)
    required_cols = ["kx", "ky", "E_minus", "E_plus"]
    col_names_str = string.(names(df))
    for c in required_cols
        c in col_names_str || error("Missing required column: $c")
    end

    subdf = df
    for (k, v) in fixed_variables
        string(k) in col_names_str || error("Filter key $(k) is not a DataFrame column.")
        col = subdf[!, k]
        if v isa Real && eltype(col) <: Real
            mask = abs.(Float64.(col) .- Float64(v)) .<= Float64(atol)
            subdf = subdf[mask, :]
        else
            mask = col .== v
            subdf = subdf[mask, :]
        end
    end

    nrow(subdf) > 0 || error("No rows matched the requested fixed variables. Try increasing atol or checking available parameter values.")

    kx_vals = sort(unique(subdf.kx))
    ky_vals = sort(unique(subdf.ky))
    nx = length(kx_vals)
    ny = length(ky_vals)

    ix = Dict(kx_vals[i] => i for i in eachindex(kx_vals))
    iy = Dict(ky_vals[j] => j for j in eachindex(ky_vals))

    Emin = fill(NaN, nx, ny)
    Emax = fill(NaN, nx, ny)

    for row in eachrow(subdf)
        i = ix[row.kx]
        j = iy[row.ky]
        Emin[i, j] = row.E_minus
        Emax[i, j] = row.E_plus
    end

    # Combined plot: both bands overlaid on the same 3D surface
    plt = surface(
        kx_vals, ky_vals, Emin';
        xlabel="k_x", ylabel="k_y", zlabel="E",
        color=:viridis,
        alpha=alpha,
        legend=false
    )

    surface!(
        plt,
        kx_vals, ky_vals, Emax';
        color=:plasma,
        alpha=alpha,
        legend=false,
    )

    label_text = isempty(fixed_variables) ? "All rows" : join(["$(k)=$(v)" for (k, v) in fixed_variables], ", ")
    plot!(plt; plot_title="QWZ Band Structure (Combined) " * label_text, size=(1000, 700))
    display(plt)
    savefig(plt, filename)
    # return plt, subdf
end

## plots the qwz bandstructure for a controllable slice in kx or ky (E+ and E- combined in same plot)
function plt_qwz_bandstructure_slice(
    df::DataFrame;
    atol::Real=1e-8,
    filename::String="qwz_bandstructure_slice.png",
    fixed_variables...
    )
    # Verify required columns exist (using string comparison for robustness)
    required_cols = ["kx", "ky", "E_minus", "E_plus"]
    col_names_str = string.(names(df))
    for c in required_cols
        c in col_names_str || error("Missing required column: $c")
    end

    subdf = df
    for (k, v) in fixed_variables
        string(k) in col_names_str || error("Filter key $(k) is not a DataFrame column.")
        col = subdf[!, k]
        if v isa Real && eltype(col) <: Real
            col_vals = Float64.(col)
            chosen = col_vals[argmin(abs.(col_vals .- Float64(v)))]
            if abs(chosen - Float64(v)) > Float64(atol)
                @warn "Snapping $(k)=$(v) to nearest available grid value $(chosen)"
            end
            subdf = subdf[col_vals .== chosen, :]
        else
            mask = col .== v
            subdf = subdf[mask, :]
        end
    end

    nrow(subdf) > 0 || error("No rows matched the requested fixed variables. Try increasing atol or checking available parameter values.")

    fixed_map = Dict(fixed_variables)
    kx_fixed = haskey(fixed_map, :kx)
    ky_fixed = haskey(fixed_map, :ky)
    (kx_fixed ⊻ ky_fixed) || error("Specify exactly one of kx or ky as the fixed momentum coordinate.")

    fixed_axis = kx_fixed ? :kx : :ky
    free_axis = kx_fixed ? :ky : :kx

    axis_vals = sort(unique(subdf[!, free_axis]))

    Eminus = similar(axis_vals, Float64)
    Eplus = similar(axis_vals, Float64)

    for (idx, val) in pairs(axis_vals)
        row = subdf[subdf[!, free_axis] .== val, :]
        nrow(row) == 1 || error("Expected exactly one row for $(free_axis)=$(val), but found $(nrow(row)).")
        Eminus[idx] = row.E_minus[1]
        Eplus[idx] = row.E_plus[1]
    end

    fixed_desc = join(["$(k)=$(v)" for (k, v) in fixed_map], ", ")
    xlabel_text = String(free_axis)
    title_text = "QWZ slice: fixed $(fixed_axis) | $(fixed_desc)"

    plt = plot(
        axis_vals, Eminus;
        xlabel=xlabel_text,
        ylabel="E",
        label="E_-",
        color=:dodgerblue3,
        linewidth=2.5,
        title=title_text,
        legend=:topright,
        ylim=(-4.1, 4.1),
    )
    plot!(
        plt,
        axis_vals, Eplus;
        label="E_+",
        color=:crimson,
        linewidth=2.5
    )

    hline!(plt, [0.0]; linestyle=:dash, color=:black, linewidth=1, label=false)
    display(plt)
    savefig(plt, filename)
    # return plt, subdf
end

In [ ]:
# # plot for
# A = 1.0
# B = 5.0
# m = 0.0

# foldername = "plots/split_bs"
# isdir(foldername) || mkdir(foldername)
# filename = joinpath(foldername, "qwz_bandstructure_split_m$(m)_A$(A)_B$(B).png")

# plt_qwz_bandstructure_split(qwz_df; filename=filename, m=m, A=A, B=B)

In [ ]:
# # plot for
# A = 1.0
# B = 1.0
# m = 1.0

# foldername = "plots/combined_bs"
# isdir(foldername) || mkdir(foldername)
# filename = joinpath(foldername, "qwz_bandstructure_combined_m$(m)_A$(A)_B$(B).png")

# plt_qwz_bandstructure_combined(qwz_df; filename=filename, m=m, A=A, B=B)

In [ ]:
# # plot for
# A = 1.0
# B = 1.0
# m = 0.0

# foldername = "plots/sliced_bs"
# isdir(foldername) || mkdir(foldername)
# filename = joinpath(foldername, "qwz_bandstructure_slice_m$(m)_A$(A)_B$(B).png")

# plt_qwz_bandstructure_slice(qwz_df; filename=filename, m=m, A=A, B=B, kx=0)

## Chern Number Calculations

### Direct analytical

In [ ]:
## Analytic topological classification of the QWZ model

function classify_qwz_topology_anal(
    ;
    B::Real=1.0,
    m::Real=0.0,
    band::Symbol=:lower
    )::Union{Missing, Int8}

    mF = Float64(m)
    BF = Float64(B)

    # Gap-closing phase boundaries (Chern number is not defined exactly here).
    if isapprox(mF, 0.0; atol=1e-12) || isapprox(mF + 2.0 * BF, 0.0; atol=1e-12) || isapprox(mF + 4.0 * BF, 0.0; atol=1e-12)
        return missing
    end

    # Lower-band QWZ Chern number for dz = m + B(2 - cos(kx) - cos(ky)).
    c_lower = Int8(round(0.5 * (sign(mF) + sign(mF + 4.0 * BF) - 2.0 * sign(mF + 2.0 * BF))))

    if band == :lower
        return c_lower
    elseif band == :upper
        return Int8(-c_lower)
    else
        error("Invalid band=$(band). Use :lower or :upper.")
    end
end

function annotate_qwz_topology(
    df::DataFrame;
    atol::Real=1e-8,
    nk_num::Int=51,
    band::Symbol=:lower
    )::DataFrame

    # for c in (:A, :B, :m)
    #     c in names(df) || error("DataFrame must contain column $(c)")
    # end

    params = unique(select(df, [:A, :B, :m]))
    n = nrow(params)

    chern_numbers = Vector{Union{Missing, Int8}}(undef, n)
    @showprogress for (i, row) in enumerate(eachrow(params))
        A_val = row.A
        B_val = row.B
        m_val = row.m
        chern_numbers[i] = classify_qwz_topology_anal(; B=B_val, m=m_val, band=band)
    end

    params[!, :chern_number] = chern_numbers

    return leftjoin(df, params; on=[:A, :B, :m])
end

# band_anal = :upper
# analytic_classified_df = annotate_qwz_topology(qwz_df; atol=1e-8, nk_num=nk, band=band_anal)
# println("Annotated DataFrame with $(nrow(analytic_classified_df)) rows and $(ncol(analytic_classified_df)) columns, including Chern numbers.")

### Indirect Numerical

In [ ]:
## Numerical (discretised BZ) Chern number from the sampled band-structure grid
function classify_qwz_topology_num(
    subdf::DataFrame;
    band::Symbol=:lower,
    gap_tol::Real=1e-6,
    chern_tol::Real=0.2
    )::Union{Missing, Int8}
    nrow(subdf) > 0 || error("classify_qwz_topology_num received an empty DataFrame slice.")

    # For this parameter slice, if the direct gap closes, treat Chern as undefined.
    min_gap = minimum(subdf.E_plus .- subdf.E_minus)
    if min_gap <= Float64(gap_tol)
        return missing
    end

    A = Float64(subdf.A[1])
    B = Float64(subdf.B[1])
    m = Float64(subdf.m[1])

    kx_vals = sort(unique(subdf.kx))
    ky_vals = sort(unique(subdf.ky))
    nx = length(kx_vals)
    ny = length(ky_vals)
    nx * ny == nrow(subdf) || error("Expected a full k-grid per (A,B,m), but found missing/duplicate points.")

    ix = Dict(kx_vals[i] => i for i in eachindex(kx_vals))
    iy = Dict(ky_vals[j] => j for j in eachindex(ky_vals))

    nhat = Array{Float64, 3}(undef, 3, nx, ny)
    @showprogress for row in eachrow(subdf)
        i = ix[row.kx]
        j = iy[row.ky]
        dx, dy, dz = dvec_qwz(row.kx, row.ky; A=A, B=B, m=m)
        nrm = sqrt(dx * dx + dy * dy + dz * dz)
        if nrm <= Float64(gap_tol)
            return missing
        end
        nhat[1, i, j] = dx / nrm
        nhat[2, i, j] = dy / nrm
        nhat[3, i, j] = dz / nrm
    end

    tri_solid_angle(a, b, c) = 2.0 * atan(dot(a, cross(b, c)), 1.0 + dot(a, b) + dot(b, c) + dot(c, a))

    omega_sum = 0.0
    @showprogress for i in 1:nx
        ip = (i == nx) ? 1 : (i + 1)
        for j in 1:ny
            jp = (j == ny) ? 1 : (j + 1)

            n00 = view(nhat, :, i, j)
            n10 = view(nhat, :, ip, j)
            n11 = view(nhat, :, ip, jp)
            n01 = view(nhat, :, i, jp)

            # Two oriented triangles per plaquette
            omega_sum += tri_solid_angle(n00, n10, n11)
            omega_sum += tri_solid_angle(n00, n11, n01)
        end
    end

    chern = omega_sum / (4.0 * pi)
    if band == :upper
        chern = -chern
    elseif band != :lower
        error("Invalid band=$(band). Use :lower or :upper.")
    end

    chern_int = round(Int, chern)
    if abs(chern - chern_int) > Float64(chern_tol)
        @warn "Numerical Chern not close to integer for (A=$(A), B=$(B), m=$(m)): C=$(chern). Rounding to $(chern_int)."
    end

    return Int8(chern_int)
end

function annotate_qwz_topology_numerical(
    df::DataFrame;
    band::Symbol=:lower,
    gap_tol::Real=1e-6,
    chern_tol::Real=0.2
    )::DataFrame
    required = [:A, :B, :m, :kx, :ky, :E_minus, :E_plus]
    present = Symbol.(names(df))
    for c in required
        c in present || error("DataFrame must contain column $(c). Available columns: $(present)")
    end

    grouped = groupby(df, [:A, :B, :m])
    ngrp = length(grouped)

    Acol = Vector{Float64}(undef, ngrp)
    Bcol = Vector{Float64}(undef, ngrp)
    mcol = Vector{Float64}(undef, ngrp)
    chern_col = Vector{Union{Missing, Int8}}(undef, ngrp)

    @showprogress for (idx, g) in enumerate(grouped)
        Acol[idx] = Float64(g.A[1])
        Bcol[idx] = Float64(g.B[1])
        mcol[idx] = Float64(g.m[1])
        chern_col[idx] = classify_qwz_topology_num(
            DataFrame(g);
            band=band,
            gap_tol=gap_tol,
            chern_tol=chern_tol
        )
    end

    params = DataFrame(A=Acol, B=Bcol, m=mcol, chern_number=chern_col)
    return leftjoin(df, params; on=[:A, :B, :m])
end

# band_num = :upper
# numerical_classified_df = annotate_qwz_topology_numerical(
#     qwz_df;
#     band=band_num,
#     gap_tol=1e-6,
#     chern_tol=0.2
# )
# println("Numerically annotated DataFrame with $(nrow(numerical_classified_df)) rows and $(ncol(numerical_classified_df)) columns.")

## Plotting Chern Number

In [ ]:
## Generic plotting of an invariant versus chosen tuning parameters
function plt_qwz_invariant(
    df::DataFrame;
    invariant_col::Symbol=:chern_number,
    x::Symbol=:m,
    y::Union{Nothing, Symbol}=nothing,
    atol::Real=1e-8,
    drop_missing::Bool=true,
    filename::String="qwz_invariant_plot.png",
    fixed_variables...
    )
    col_names = Symbol.(names(df))
    invariant_col in col_names || error("Missing invariant column: $(invariant_col). Available columns: $(col_names)")
    x in col_names || error("Missing x column: $(x). Available columns: $(col_names)")
    if y !== nothing
        y in col_names || error("Missing y column: $(y). Available columns: $(col_names)")
    end

    subdf = df
    for (k, v) in fixed_variables
        k in col_names || error("Filter key $(k) is not a DataFrame column.")
        col = subdf[!, k]
        if v isa Real && eltype(skipmissing(col)) <: Real
            col_vals = Float64.(collect(skipmissing(col)))
            isempty(col_vals) && error("Column $(k) has no numeric values after missing removal.")
            chosen = col_vals[argmin(abs.(col_vals .- Float64(v)))]
            if abs(chosen - Float64(v)) > Float64(atol)
                @warn "Snapping $(k)=$(v) to nearest available grid value $(chosen)"
            end
            subdf = subdf[abs.(Float64.(subdf[!, k]) .- chosen) .<= Float64(atol), :]
        else
            subdf = subdf[subdf[!, k] .== v, :]
        end
    end

    nrow(subdf) > 0 || error("No rows matched the requested fixed variables.")
    if drop_missing
        subdf = subdf[.!ismissing.(subdf[!, invariant_col]), :]
    end
    nrow(subdf) > 0 || error("No non-missing invariant values available after filtering.")

    if y === nothing
        xvals = sort(unique(subdf[!, x]))
        inv_vals = Vector{Float64}(undef, length(xvals))

        for (i, xv) in pairs(xvals)
            rows = subdf[subdf[!, x] .== xv, :]
            vals = Float64.(collect(skipmissing(rows[!, invariant_col])))
            isempty(vals) && (inv_vals[i] = NaN; continue)
            inv_vals[i] = first(vals)
            if !all(isapprox.(vals, inv_vals[i]; atol=Float64(atol)))
                @warn "Multiple invariant values found for $(x)=$(xv); using average value."
                inv_vals[i] = sum(vals) / length(vals)
            end
        end

        plt = plot(
            xvals, inv_vals;
            xlabel=String(x),
            ylabel=String(invariant_col),
            title="Invariant vs $(x), fixed $(join(["$(k)=$(v)" for (k, v) in fixed_variables], ", "))",
            marker=:circle,
            linewidth=2.5,
            color=:navy,
            legend=false
        )
        hline!(plt, [0.0]; linestyle=:dash, color=:black, linewidth=1, label=false)
        display(plt)
        savefig(plt, filename)
        return plt
    else
        xvals = sort(unique(subdf[!, x]))
        yvals = sort(unique(subdf[!, y]))
        Z = fill(NaN, length(yvals), length(xvals))

        x_index = Dict(val => i for (i, val) in pairs(xvals))
        y_index = Dict(val => j for (j, val) in pairs(yvals))

        for row in eachrow(subdf)
            i = x_index[row[x]]
            j = y_index[row[y]]
            val = row[invariant_col]
            Z[j, i] = ismissing(val) ? NaN : Float64(val)
        end

        plt = heatmap(
            xvals, yvals, Z;
            xlabel=String(x),
            ylabel=String(y),
            title="$(invariant_col) in $(x)-$(y) space, fixed $(join(["$(k)=$(v)" for (k, v) in fixed_variables], ", "))",
            color=:RdBu,
            clims=(-1.0, 1.0),
            colorbar_title=String(invariant_col)
        )
        display(plt)
        savefig(plt, filename)
        return plt
    end
end

## Convenience wrapper for topology-specific plots
function plt_qwz_topology(
    df::DataFrame;
    x::Symbol=:m,
    y::Union{Nothing, Symbol}=:B,
    filename::String="qwz_topology_plot.png",
    fixed_variables...
    )
    return plt_qwz_invariant(
        df;
        invariant_col=:chern_number,
        x=x,
        y=y,
        filename=filename,
        fixed_variables...
    )
end

### Plotting Analytical C

In [ ]:
# foldername = "plots/analytic_topo"
# isdir(foldername) || mkdir(foldername)

# # 2D phase diagram (chern_number vs m and B, fixing A)
# A = 1.0
# filename = joinpath(foldername, "qwz_analytic_topology_phase_diagram_A$(A)_$(band_anal).png")
# plt_qwz_topology(analytic_classified_df; filename=filename, x=:m, y=:B, A=1.0)

# # 1D cut (chern_number vs m, fixing A and B)
# A = 1.0
# B = 0.5
# filename = joinpath(foldername, "qwz_analytic_topology_cut_A$(A)_B$(B).png")
# plt_qwz_topology(analytic_classified_df; filename=filename, x=:m, y=nothing, A=A, B=B)


# println("Plotted QWZ topology phase diagrams and cuts based on the classified DataFrame.")

### Plotting Numerical C

In [ ]:
# foldername = "plots/numerical_topo"
# isdir(foldername) || mkdir(foldername)

# # 2D phase diagram (chern_number vs m and B, fixing A)
# A = 1.0
# filename = joinpath(foldername, "qwz_numerical_topology_phase_diagram_A$A.png")
# plt_qwz_topology(numerical_classified_df; filename=filename, x=:m, y=:B, A=1.0)

# # 1D cut (chern_number vs m, fixing A and B)
# A = 1.0
# B = 0.5
# filename = joinpath(foldername, "qwz_numerical_topology_cut_A$(A)_B$(B).png")
# plt_qwz_topology(numerical_classified_df; filename=filename, x=:m, y=nothing, A=A, B=B)


# println("Plotted QWZ topology phase diagrams and cuts based on the classified DataFrame.")

## Spectral Localiser 

### Hamiltonian (real space)

For the k-space model used above, the real-space QWZ Hamiltonian on a square lattice has one two-component orbital/spinor per site.

The correct tight-binding decomposition for the model implemented in this notebook is:
- on-site block: (m + 2B) σ_z
- hopping in +x: T_x = -(B/2) σ_z - (iA/2) σ_x
- hopping in +y: T_y = -(B/2) σ_z - (iA/2) σ_y

So your derivation is accurate in structure, but the general coefficient is m + 2B on-site, not just M unless you define M = m + 2B. The diagonal hopping terms also keep the factor B. For B = 1, your formulas reduce to the familiar M = m + 2 form.

#### Generation

In [ ]:
# Real-space block coefficients for the QWZ model.
function real_space_qwz_blocks(; A::Real=1.0, B::Real=1.0, m::Real=0.0)
    onsite = ComplexF64.((m + 2.0 * B) * sigma_z)
    tx = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_x)
    ty = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_y)
    return onsite, tx, ty
end

# Map a site and orbital to a matrix index in the 2LxLy basis.
site_index_qwz(x::Int, y::Int, orb::Int, Lx::Int, Ly::Int) = 2 * ((y - 1) * Lx + (x - 1)) + orb

function real_space_hamiltonian_qwz(
    Lx::Int,
    Ly::Int;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    periodic_x::Bool=false,
    periodic_y::Bool=false,
    sparse_output::Bool=true
    )
    Lx > 0 || error("Lx must be positive.")
    Ly > 0 || error("Ly must be positive.")

    onsite, tx, ty = real_space_qwz_blocks(; A=A, B=B, m=m)
    nsites = Lx * Ly
    dim = 2 * nsites

    H = sparse_output ? spzeros(ComplexF64, dim, dim) : zeros(ComplexF64, dim, dim)

    function add_block!(mat, row_site::Tuple{Int, Int}, col_site::Tuple{Int, Int}, block::AbstractMatrix{<:Number})
        (xr, yr) = row_site
        (xc, yc) = col_site
        row_base = site_index_qwz(xr, yr, 1, Lx, Ly)
        col_base = site_index_qwz(xc, yc, 1, Lx, Ly)
        @inbounds for a in 0:1, b in 0:1
            mat[row_base + a, col_base + b] += ComplexF64(block[a + 1, b + 1])
        end
        return nothing
    end

    for y in 1:Ly, x in 1:Lx
        add_block!(H, (x, y), (x, y), onsite)

        if x < Lx
            add_block!(H, (x + 1, y), (x, y), tx)
            add_block!(H, (x, y), (x + 1, y), tx')
        elseif periodic_x
            add_block!(H, (1, y), (x, y), tx)
            add_block!(H, (x, y), (1, y), tx')
        end

        if y < Ly
            add_block!(H, (x, y + 1), (x, y), ty)
            add_block!(H, (x, y), (x, y + 1), ty')
        elseif periodic_y
            add_block!(H, (x, 1), (x, y), ty)
            add_block!(H, (x, y), (x, 1), ty')
        end
    end

    return H
end

# Convenience dense version for small lattices or inspection.
function real_space_hamiltonian_qwz_dense(
    Lx::Int,
    Ly::Int;
    kwargs...
    )
    return Matrix(real_space_hamiltonian_qwz(Lx, Ly; kwargs...))
end

#### Validation

##### Validation 1: Reconstruction & Hermicity Error

In [ ]:
## Validation 1: Fourier reconstruction of the Bloch Hamiltonian from the real-space blocks.

## Choose arbitrary parameters and momentum point for testing.
A = 1.2
B = 0.7
m = -0.4
kx = 0.37
ky = -1.13

# Helper: form the 2x2 Bloch Hamiltonian from the real-space blocks (uses the same blocks as the lattice builder).
function real_space_bloch_hamiltonian(kx::Real, ky::Real; A::Real=1.0, B::Real=1.0, m::Real=0.0)
    onsite, tx, ty = real_space_qwz_blocks(; A=A, B=B, m=m)
    return onsite + tx * exp(im * kx) + tx' * exp(-im * kx) + ty * exp(im * ky) + ty' * exp(-im * ky)
end

H_from_real_space = real_space_bloch_hamiltonian(kx, ky; A=A, B=B, m=m)
H_from_kspace = hamiltonian_qwz(kx, ky; A=A, B=B, m=m)

reconstruction_error = norm(H_from_real_space - H_from_kspace)
hermiticity_error = norm(H_from_real_space - H_from_real_space')

println("Reconstruction error = ", reconstruction_error)
println("Hermiticity error = ", hermiticity_error)

# heatmap(
#     real.(H_from_real_space);
#     aspect_ratio=:equal,
#     color=:viridis,
#     title="Reconstructed H(k) real part",
#     xlabel="matrix column",
#     ylabel="matrix row",
#     colorbar=false
#  )

In [ ]:
## Search of reconstruction and hermicity error over the full k and parameter space (A, B, m) to validate the real-space construction.

A_vals = [1.0, 2.0, 5.0]
B_vals = collect(0.0:0.2:2.0)
m_vals = collect(-8.0:0.1:2.0)
kx_vals = collect(range(-pi, pi; length=21))[1:end-1]
ky_vals = collect(range(-pi, pi; length=21))[1:end-1]

error_df = DataFrame(A=Float64[], B=Float64[], m=Float64[], kx=Float64[], ky=Float64[], recon_err=Float64[], herm_err=Float64[], tol_pass=Bool[])

@showprogress for A in A_vals, B in B_vals, m in m_vals
    for kx in kx_vals, ky in ky_vals
        H_real = real_space_bloch_hamiltonian(kx, ky; A=A, B=B, m=m)
        H_k = hamiltonian_qwz(kx, ky; A=A, B=B, m=m)
        recon_err = norm(H_real - H_k)
        herm_err = norm(H_real - H_real')
        if recon_err > 1e-12 || herm_err > 1e-12
            tol_pass=false
        else
            tol_pass=true
        end
        push!(error_df, (A, B, m, kx, ky, recon_err, herm_err, tol_pass))
    end
end

if all(error_df.tol_pass)
    println("All tests passed: real-space reconstruction and hermiticity validated across the parameter space.")
else
    nfail = count(!, error_df.tol_pass)
    println("Warning: $(nfail) tests failed out of $(nrow(error_df)). Check error_df for details.")
end

##### Validation 2: PBC vs analytic error

In [ ]:
# Validation 2: periodic-boundary spectrum versus the analytic Bloch spectrum.
A = 1.0
B = 1.0
m = -2.1
Lx = 8
Ly = 8

H_pbc = real_space_hamiltonian_qwz(Lx, Ly; A=A, B=B, m=m, periodic_x=true, periodic_y=true, sparse_output=false)
eigvals_pbc = sort(real(eigvals(H_pbc)))

kx_vals = [2pi * n / Lx for n in 0:Lx-1]
ky_vals = [2pi * n / Ly for n in 0:Ly-1]
eigvals_bloch = Float64[]
for kx in kx_vals, ky in ky_vals
    e1, e2 = band_energies_qwz(kx, ky; A=A, B=B, m=m)
    push!(eigvals_bloch, e1)
    push!(eigvals_bloch, e2)
end
eigvals_bloch = sort(eigvals_bloch)

comparison_error = maximum(abs.(eigvals_pbc .- eigvals_bloch))
println("Maximum periodic-spectrum mismatch = ", comparison_error)

p1 = plot(
    eigvals_bloch;
    label="analytic Bloch spectrum",
    linewidth=2.5,
    color=:black,
    xlabel="sorted state index",
    ylabel="energy",
    title="PBC spectrum comparison"
 )
plot!(p1, eigvals_pbc; label="real-space PBC", linewidth=1.5, linestyle=:dash, color=:dodgerblue3)

p2 = plot(
    abs.(eigvals_pbc .- eigvals_bloch);
    label="|ΔE|",
    linewidth=2.5,
    color=:crimson,
    xlabel="sorted state index",
    ylabel="absolute error",
    title="PBC spectral error"
 )

plot(p1, p2; layout=(1, 2), size=(800, 600))

In [ ]:
## Search of PBC error over the full k and parameter space (A, B, m) to validate the real-space construction.

A_vals = [1.0, 2.0, 5.0]
B_vals = collect(0.0:0.2:2.0)
m_vals = collect(-8.0:0.1:2.0)
Lx_vals = [4, 6, 8]
Ly_vals = [4, 6, 8]

error_df = DataFrame(A=Float64[], B=Float64[], m=Float64[], Lx=Int[], Ly=Int[], comparison_error=Float64[], tol_pass=Bool[])

@showprogress for A in A_vals, B in B_vals, m in m_vals
    for Lx in Lx_vals, Ly in Ly_vals
        H_pbc = real_space_hamiltonian_qwz(Lx, Ly; A=A, B=B, m=m, periodic_x=true, periodic_y=true, sparse_output=false)
        eigvals_pbc = sort(real(eigvals(H_pbc)))

        kx_vals = [2pi * n / Lx for n in 0:Lx-1]
        ky_vals = [2pi * n / Ly for n in 0:Ly-1]
        eigvals_bloch = Float64[]
        for kx in kx_vals, ky in ky_vals
            e1, e2 = band_energies_qwz(kx, ky; A=A, B=B, m=m)
            push!(eigvals_bloch, e1)
            push!(eigvals_bloch, e2)
        end
        eigvals_bloch = sort(eigvals_bloch)

        comparison_error = maximum(abs.(eigvals_pbc .- eigvals_bloch))

        if comparison_error > 1e-12
            tol_pass=false
        else
            tol_pass=true
        end
        push!(error_df, (A, B, m, Lx, Ly, comparison_error, tol_pass))
    end
end

if all(error_df.tol_pass)
    println("All tests passed: real-space reconstruction with PBC validated across the parameter space.")
else
    nfail = count(!, error_df.tol_pass)
    println("Warning: $(nfail) tests failed out of $(nrow(error_df)). Check error_df for details.")
end

##### Validation 3: OBC inspection

In [ ]:
# Validation 3: open-boundary spectrum and edge localization in a topological phase.
A = 1.0
B = 1.0
m = -1.0
Lx = 25
Ly = 25

Lxs = [5,10,15,20,25]
Lys = [5,10,15,20,25]

@showprogress for (Lx, Ly) in zip(Lxs, Lys)
    H_obc = real_space_hamiltonian_qwz(Lx, Ly; A=A, B=B, m=m, periodic_x=false, periodic_y=true, sparse_output=false)
    hermiticity_error_obc = norm(H_obc - H_obc')
    println("OBC Hermiticity error = ", hermiticity_error_obc)

    F = eigen(H_obc)
    eigs = real(F.values)
    order = sortperm(abs.(eigs))
    eigs_sorted = eigs[sortperm(eigs)]

    edge_idx = order[1]
    psi = F.vectors[:, edge_idx]
    edge_density = zeros(Float64, Lx, Ly)
    for y in 1:Ly, x in 1:Lx
        base = 2 * ((y - 1) * Lx + (x - 1))
        edge_density[x, y] = abs2(psi[base + 1]) + abs2(psi[base + 2])
    end

    p1 = scatter(
        1:length(eigs_sorted), eigs_sorted;
        label="OBC eigenvalues",
        markerstrokewidth=0,
        markersize=3,
        color=:slateblue,
        xlabel="sorted state index",
        ylabel="energy",
        title="Open-boundary spectrum"
    )
    hline!(p1, [0.0]; linestyle=:dash, color=:black, label=false)

    p2 = heatmap(
        1:Lx, 1:Ly, edge_density';
        xlabel="x", ylabel="y",
        title="Lowest-|E| eigenstate density",
        color=:viridis,
        aspect_ratio=1,
        colorbar_title="|ψ|²"
    )

    plot(p1, p2; layout=(1, 2), size=(800, 600))

    foldername = joinpath("plots", "obc_spec", "m$(m)")
    isdir(foldername) || mkpath(foldername)

    savefig(joinpath(foldername, "qwz_pbcy_obc_edge_state_A$(A)_B$(B)_m$(m)_Lx$(Lx)_Ly$(Ly).png"))
end

### SpecLoc Calc

#### Pre-Compute Steps

In [ ]:
## Estimate bulk bandgap


## Check valid E-range (in bandgap) for the given parameters.


## Estimate kappa ~ Δ/R

function estimate_kappa_for_params(Lx::Int, Ly::Int, Avals, Bvals, mvals; nk::Int=101)
    ks = collect(range(-pi, pi; length=nk + 1))[1:end-1]
    rows = NamedTuple[]
    R = max(Lx, Ly) / 2.0
    for A in Avals, B in Bvals, m in mvals
        e_minus_max = -Inf
        e_plus_min = Inf
        min_gap = Inf
        for kx in ks, ky in ks
            e1, e2 = band_energies_qwz(kx, ky; A=A, B=B, m=m)   # e1 <= e2
            min_gap = min(min_gap, e2 - e1)
            e_minus_max = max(e_minus_max, e1)
            e_plus_min  = min(e_plus_min,  e2)
        end
        # is E=0 inside a hard gap?
        E0_in_gap = (e_minus_max < 0.0) && (e_plus_min > 0.0)
        kappa0 = (min_gap <= 0.0) ? 0.0 : (min_gap / R)
        kappa_scan = if kappa0 > 0.0
            (kappa0 * 1e-2, kappa0 * 1e2)
        else
            (1e-6, 1e+1)    # fallback if gap closed or negative
        end
        push!(rows, (A=Float64(A), B=Float64(B), m=Float64(m),
                     min_gap=min_gap, R=R, kappa0=kappa0,
                     kappa_scan_min=kappa_scan[1], kappa_scan_max=kappa_scan[2],
                     E0_in_gap=E0_in_gap))
    end
    return DataFrame(rows)
end

# # Usage example (adapt Lx,Ly and parameter lists as you like):
# Lx, Ly = 10, 10
# Avals = [1.0]
# Bvals = [0.5, 1.0, 2.0]
# mvals = [-2.0, 0.0, 2.0]
# kappa_est_df = estimate_kappa_for_params(Lx, Ly, Avals, Bvals, mvals; nk=121)
# println(kappa_est_df)
# # Optional: save results
# # CSV.write("kappa_estimates.csv", kappa_est_df)

#### Compute

##### Compute 1: slow

In [ ]:
# Build position operators (x and y) that act on the full 2-orbital per site basis.
function build_position_operators(
    Lx::Int, 
    Ly::Int
    )

    nsites = Lx * Ly
    dim = 2 * nsites
    xvec = zeros(Float64, dim)
    yvec = zeros(Float64, dim)
    for y in 1:Ly, x in 1:Lx
        base = 2 * ((y - 1) * Lx + (x - 1))
        xvec[base + 1] = x
        xvec[base + 2] = x
        yvec[base + 1] = y
        yvec[base + 2] = y
    end
    X = Diagonal(ComplexF64.(xvec))
    Y = Diagonal(ComplexF64.(yvec))
    return X, Y
end

# Compute the spectral localiser gap value (minimum absolute eigenvalue of the localiser)
# and the signature of the localiser. The signature is (#positive eigenvals - #negative eigenvals).
# H: full real-space Hamiltonian (2*nsites × 2*nsites)
# X, Y: position operators (Diagonal matrices matching H dimension)
# x0, y0: position at which to center the localiser (use site coordinates 1..Lx,1..Ly)
# E: energy around which the localiser is computed
# kappa: length-scale weighting of position terms
function spectral_localiser_value(
    H::AbstractMatrix{ComplexF64}, 
    X::Diagonal, 
    Y::Diagonal, 
    x0::Real, 
    y0::Real, 
    E::Real; 
    kappa::Real=1.0
    )::Float64

    D = size(H, 1)
    I_D = Matrix{ComplexF64}(I, D, D)
    # Convert Diagonal operators to dense so block construction is straightforward
    Xmat = Matrix(X)
    Ymat = Matrix(Y)

    Hshift = H - E * I_D
    Ablock = kappa * (Xmat - x0 * I_D)
    Bblock = kappa * (Ymat - y0 * I_D)

    # Construct the Hermitian localiser L (2D × 2D blocks)
    L = [Hshift  Ablock - im * Bblock;
         Ablock + im * Bblock  -Hshift]

    vals = eigvals(Hermitian(L))
    return minimum(abs.(real(vals)))
end

# Compute the signature (n_pos - n_neg) and also return the minimum absolute eigenvalue.
# If eigenvalues lie within `zero_tol` of zero they are considered zero.
function spectral_localiser_signature(
    H::AbstractMatrix{ComplexF64},
    X::Diagonal,
    Y::Diagonal,
    x0::Real,
    y0::Real,
    E::Real; 
    kappa::Real=1.0,
    zero_tol::Real=1e-12
    )::Tuple{Int, Float64}

    D = size(H, 1)
    I_D = Matrix{ComplexF64}(I, D, D)
    Xmat = Matrix(X)
    Ymat = Matrix(Y)

    Hshift = H - E * I_D
    Ablock = kappa * (Xmat - x0 * I_D)
    Bblock = kappa * (Ymat - y0 * I_D)

    L = [Hshift  Ablock - im * Bblock;
         Ablock + im * Bblock  -Hshift]

    vals = eigvals(Hermitian(L))
    revals = real(vals)
    npos = count(>(zero_tol), revals)
    nneg = count(<(-zero_tol), revals)
    minabs = minimum(abs.(revals))
    signature = npos - nneg
    return signature, minabs
end

# Compute the Chern number from the localiser signature: C = 1/2 * signature.
# Returns `missing` if there are near-zero eigenvalues or if the signature/2 is not (near-)integer.
function chern_from_localiser_signature(signature::Int, minabs::Real; zero_tol::Real=1e-12, chern_tol::Real=1e-6)
    if minabs <= zero_tol
        return missing
    end
    chern_d = signature / 2
    chern_round = round(Int, chern_d)
    if abs(chern_d - chern_round) > chern_tol
        return missing
    end
    return Int8(chern_round)
end

# Scan the spectral localiser over parameter ranges, positions and energies and return a DataFrame
function slow_compute_spectral_localiser_df(
    Lx::Int, 
    Ly::Int; 
    Avals::AbstractVector{<:Real} = [1.0], 
    Bvals::AbstractVector{<:Real} = [1.0], 
    mvals::AbstractVector{<:Real} = [0.0],
    xs::AbstractVector = collect(1:Lx), 
    ys::AbstractVector = collect(1:Ly), 
    Es::AbstractVector = [0.0],
    kappas::AbstractVector{<:Real} = [1.0], 
    periodic_x::Bool=false, 
    periodic_y::Bool=false, 
    sparse_output::Bool=false
)::DataFrame

    rows = Vector{NamedTuple{(:A, :B, :m, :Lx, :Ly, :x, :y, :E, :kappa, :localiser_gap, :signature, :chern_number),Tuple{Float64,Float64,Float64,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Int64,Union{Missing, Int8}}}}()

    total_iter = length(Avals) * length(Bvals) * length(mvals) * length(xs) * length(ys) * length(Es)
    # idx = 0
    @showprogress for A in Avals, B in Bvals, m in mvals, x0 in xs, y0 in ys, E in Es, kappa in kappas
        # Build H and position operators for this parameter set
        H = real_space_hamiltonian_qwz(Lx, Ly; A=A, B=B, m=m, periodic_x=periodic_x, periodic_y=periodic_y, sparse_output=sparse_output)
        X, Y = build_position_operators(Lx, Ly)

        # for x0 in xs, y0 in ys, E in Es, kappa in kappas
        sig, minabs = spectral_localiser_signature(H, X, Y, x0, y0, E; kappa=kappa)
        chern = chern_from_localiser_signature(sig, minabs; zero_tol=1e-12, chern_tol=1e-6)
        push!(rows, (A=Float64(A), B=Float64(B), m=Float64(m), Lx=Lx, Ly=Ly, x=Float64(x0), y=Float64(y0), E=Float64(E), kappa=Float64(kappa), localiser_gap=Float64(minabs), signature=Int64(sig), chern_number=chern))
        # idx += 1
        # end
    end

    return DataFrame(rows)
end

In [ ]:
Avals = [1.0]
Bvals = [1.0]
mvals = [1.0]
Lx = 10
Ly = 10
xs = collect(1:Lx)
ys = collect(1:Ly)
Es = [0.0]
kappa = [1.0]

slow_localiser_df = slow_compute_spectral_localiser_df(Lx, Ly; Avals=Avals, Bvals=Bvals, mvals=mvals, xs=xs, ys=ys, Es=Es, kappas=kappa, periodic_x=false, periodic_y=false, sparse_output=false)
println("Computed spectral localiser DataFrame with $(nrow(slow_localiser_df)) rows and $(ncol(slow_localiser_df)) columns.")

##### Compute 2: fast

i) Topological protection measure, the spectral localiser `gap'
$$ \mu^C=\text{min}|\lambda(L)|$$
This can be computed efficiently suing standard sparse matrix lowest eval only methods

ii) Topological invariant, the signature of the diagonalised matrix
$$C(x,y,E) = \frac{1}{2} \text{sig}(L)$$
This can be computed efficiently usnig the sparse $LDL^T$ method.

In [ ]:
## Efficient localiser gap compute
function compute_localiser_gap_Krylov(
    L::SparseMatrixCSC{ComplexF64, Int};
    kk_tol::Real=1e-8,
    kk_maxiter::Int=1000
)::Float64

    L2_action(x) = L * (L * x) # matrix-free action for L^2. 
    
    n = size(L, 1)
    
    # KrylovKit for 1 eigenvalue (:SR = Smallest Real).
    # Provide a random initial vector to kickstart the Lanczos process.
    evals, evecs, info = eigsolve(L2_action, rand(ComplexF64, n), 1, :SR; 
                                  ishermitian=true, tol=kk_tol, maxiter=kk_maxiter)
    
    localiser_gap = sqrt(abs(evals[1]))
    
    return localiser_gap
end

## Efficient chern number compute
function compute_chern_signature_LDLT(
    L::SparseMatrixCSC{ComplexF64, Int}
)::Float64

    # 1. Separate the complex matrix into Real and Imaginary parts
    A = real(L)
    B = imag(L)
    
    # 2. Construct the 2N x 2N Real Symmetric Isomorphism matrix
    L_real = [A -B; 
              B  A]
              
    # 3. Perform sparse LDL^T factorization
    ldl_fact = ldl(Symmetric(L_real))
    
    # 4. Extract the diagonal matrix D (.d)
    D_diag = ldl_fact.d
    
    # 5. Count the positive and negative entries
    n_pos = count(x -> x > 0, D_diag)
    n_neg = count(x -> x < 0, D_diag)
    
    # 6. Calculate the Chern number (divided by 4 due to the real isomorphism and 1/2 sig formula)
    chern = (n_pos - n_neg) / 4.0
    
    return chern
end

## Auxilliary functions
# Build position operators (x and y) that act on the full 2-orbital per site basis.
function build_position_operators(
    Lx::Int, 
    Ly::Int
)

    nsites = Lx * Ly
    dim = 2 * nsites
    xvec = zeros(Float64, dim)
    yvec = zeros(Float64, dim)
    for y in 1:Ly, x in 1:Lx
        base = 2 * ((y - 1) * Lx + (x - 1))
        xvec[base + 1] = x
        xvec[base + 2] = x
        yvec[base + 1] = y
        yvec[base + 2] = y
    end
    X = Diagonal(ComplexF64.(xvec))
    Y = Diagonal(ComplexF64.(yvec))
    return X, Y
end

# Map a site and orbital to a matrix index in the 2LxLy basis.
site_index_qwz(x::Int, y::Int, orb::Int, Lx::Int, Ly::Int) = 2 * ((y - 1) * Lx + (x - 1)) + orb

function real_space_hamiltonian_qwz(
    Lx::Int,
    Ly::Int;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    periodic_x::Bool=false,
    periodic_y::Bool=false,
    sparse_output::Bool=true
)
    Lx > 0 || error("Lx must be positive.")
    Ly > 0 || error("Ly must be positive.")

    onsite, tx, ty = real_space_qwz_blocks(; A=A, B=B, m=m)
    nsites = Lx * Ly
    dim = 2 * nsites

    H = sparse_output ? spzeros(ComplexF64, dim, dim) : zeros(ComplexF64, dim, dim)

    function add_block!(mat, row_site::Tuple{Int, Int}, col_site::Tuple{Int, Int}, block::AbstractMatrix{<:Number})
        (xr, yr) = row_site
        (xc, yc) = col_site
        row_base = site_index_qwz(xr, yr, 1, Lx, Ly)
        col_base = site_index_qwz(xc, yc, 1, Lx, Ly)
        @inbounds for a in 0:1, b in 0:1
            mat[row_base + a, col_base + b] += ComplexF64(block[a + 1, b + 1])
        end
        return nothing
    end

    for y in 1:Ly, x in 1:Lx
        add_block!(H, (x, y), (x, y), onsite)

        if x < Lx
            add_block!(H, (x + 1, y), (x, y), tx)
            add_block!(H, (x, y), (x + 1, y), tx')
        elseif periodic_x
            add_block!(H, (1, y), (x, y), tx)
            add_block!(H, (x, y), (1, y), tx')
        end

        if y < Ly
            add_block!(H, (x, y + 1), (x, y), ty)
            add_block!(H, (x, y), (x, y + 1), ty')
        elseif periodic_y
            add_block!(H, (x, 1), (x, y), ty)
            add_block!(H, (x, y), (x, 1), ty')
        end
    end

    return H
end

# Wrapper function over parameter ranges
function compute_spec_loc_df(
    Lx::Int, 
    Ly::Int; 
    Avals::AbstractVector{<:Real} = [1.0], 
    Bvals::AbstractVector{<:Real} = [1.0], 
    mvals::AbstractVector{<:Real} = [0.0],
    xs::AbstractVector = collect(1:Lx), 
    ys::AbstractVector = collect(1:Ly), 
    Es::AbstractVector = [0.0],
    kappas::AbstractVector{<:Real} = [1.0],
    periodic_x::Bool=false, 
    periodic_y::Bool=false, 
    sparse_output::Bool=true,
    kk_tol::Real=1e-8,
    kk_maxiter::Int=1000,
    gap_tol::Real=1e-6
)::DataFrame

    # Prepare container matching your exact named tuple specifications
    rows = Vector{NamedTuple{
        (:A, :B, :m, :Lx, :Ly, :x, :y, :E, :kappa, :localiser_gap, :chern_number),
        Tuple{Float64,Float64,Float64,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Union{Missing, Float64}}
    }}()

    @showprogress for A in Avals, B in Bvals, m in mvals

        H = real_space_hamiltonian_qwz(Lx, Ly; A=A, B=B, m=m, periodic_x=periodic_x, periodic_y=periodic_y, sparse_output=sparse_output)
        X, Y = build_position_operators(Lx, Ly)

        for x0 in xs, y0 in ys, E in Es, kappa in kappas
            # 1. Shift Hamiltonian by the Fermi energy
            H_diff = sparse(H) - E * I
            
            # 2. Build sparse off-diagonal blocks using fast vector-wise math on X.diag and Y.diag.
            # Since X and Y are Diagonal types, extracting the raw `.diag` vector 
            # bypasses full matrix operations entirely!
            diag_vals = kappa .* (X.diag .- x0) .- 1im .* kappa .* (Y.diag .- y0)
            B_block = sparse(Diagonal(diag_vals))
            B_block_adj = sparse(B_block') # Converts the Adjoint wrapper to standard SparseMatrixCSC
            
            # 3. Assemble the 2x2 block Spectral localiser L (size 4N x 4N)
            L = [H_diff       B_block;
                 B_block_adj -H_diff]

            gap = compute_localiser_gap_Krylov(L; kk_tol=kk_tol, kk_maxiter=kk_maxiter)
            chern_val = compute_chern_signature_LDLT(L)
            
            # If the localiser gap closes (below gap_tol), treat Chern as ill-defined (missing)
            chern_number = if gap <= gap_tol
                missing
            else
                chern_val
            end
            
            push!(rows, (
                A=Float64(A), B=Float64(B), m=Float64(m), 
                Lx=Lx, Ly=Ly, 
                x=Float64(x0), y=Float64(y0), 
                E=Float64(E), 
                kappa=Float64(kappa),
                localiser_gap=gap,
                chern_number=chern_number
            ))
        end
    end

    return DataFrame(rows)
end

function compute_spec_loc_chern_only_df(
    Lx::Int, 
    Ly::Int; 
    Avals::AbstractVector{<:Real} = [1.0], 
    Bvals::AbstractVector{<:Real} = [1.0], 
    mvals::AbstractVector{<:Real} = [0.0],
    xs::AbstractVector = collect(1:Lx), 
    ys::AbstractVector = collect(1:Ly), 
    Es::AbstractVector = [0.0],
    kappas::AbstractVector{<:Real} = [1.0],
    periodic_x::Bool=false, 
    periodic_y::Bool=false, 
    sparse_output::Bool=true,
    kk_tol::Real=1e-8,
    kk_maxiter::Int=1000,
    gap_tol::Real=1e-6
)::DataFrame

    # Prepare container matching your exact named tuple specifications
    rows = Vector{NamedTuple{
        (:A, :B, :m, :Lx, :Ly, :x, :y, :E, :kappa, :chern_number),
        Tuple{Float64,Float64,Float64,Int64,Int64,Float64,Float64,Float64,Float64,Union{Missing, Float64}}
    }}()

    @showprogress for A in Avals, B in Bvals, m in mvals

        H = real_space_hamiltonian_qwz(Lx, Ly; A=A, B=B, m=m, periodic_x=periodic_x, periodic_y=periodic_y, sparse_output=sparse_output)
        X, Y = build_position_operators(Lx, Ly)

        for x0 in xs, y0 in ys, E in Es, kappa in kappas
            # 1. Shift Hamiltonian by the Fermi energy
            H_diff = sparse(H) - E * I
            
            # 2. Build sparse off-diagonal blocks using fast vector-wise math on X.diag and Y.diag.
            # Since X and Y are Diagonal types, extracting the raw `.diag` vector 
            # bypasses full matrix operations entirely!
            diag_vals = kappa .* (X.diag .- x0) .- 1im .* kappa .* (Y.diag .- y0)
            B_block = sparse(Diagonal(diag_vals))
            B_block_adj = sparse(B_block') # Converts the Adjoint wrapper to standard SparseMatrixCSC
            
            # 3. Assemble the 2x2 block Spectral localiser L (size 4N x 4N)
            L = [H_diff       B_block;
                 B_block_adj -H_diff]

            # gap = compute_localiser_gap_Krylov(L; kk_tol=kk_tol, kk_maxiter=kk_maxiter)
            chern_val = compute_chern_signature_LDLT(L)
            
            push!(rows, (
                A=Float64(A), B=Float64(B), m=Float64(m), 
                Lx=Lx, Ly=Ly, 
                x=Float64(x0), y=Float64(y0), 
                E=Float64(E), 
                kappa=Float64(kappa),
                chern_number=chern_val
            ))
        end
    end

    return DataFrame(rows)
end

In [ ]:
## Do kappa search at known parameters, positions and energy

Avals = [1.0]
Bvals = [1.0]
mvals = [1.0, -1.0]
Lx = 30
Ly = 30
xs = [15.5] # collect(range(-Lx, 2Lx, length=50))
ys = [15.5] # collect(range(-Ly, 2Ly, length=50)) #collect(1:Ly)
Es = [0.0]
kappas = logrange(1e-4, 1e4, 30)

kap_scan_localiser_df = compute_spec_loc_df(
    Lx, 
    Ly; 
    Avals=Avals, 
    Bvals=Bvals, 
    mvals=mvals, 
    xs=xs, 
    ys=ys, 
    Es=Es, 
    kappas=kappas, 
    periodic_x=false, 
    periodic_y=false, 
    sparse_output=true,
    kk_tol=1e-8,
    kk_maxiter=1000,
    gap_tol=1e-6
)

println("Computed spectral localiser DataFrame with $(nrow(kap_scan_localiser_df)) rows and $(ncol(kap_scan_localiser_df)) columns.")

#### Validation

##### Val 1: Fast

In [ ]:
## Plots of localiser gap and signature against kappa at single fixed position and energy (plus over varying parameters)

function plt_localiser_kappa_from_df(
    localiser_df::DataFrame; 
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    x0::Union{Nothing,Real}=nothing, 
    y0::Union{Nothing,Real}=nothing,
    E::Real=0.0,
    filename::String="plots/localiser_position_heatmaps.png"
)

    # Filter by parameters
    df = localiser_df[(localiser_df.A .== Float64(A)) .& (localiser_df.B .== Float64(B)) .& (localiser_df.m .== Float64(m)) .& (localiser_df.E .== Float64(E)), :]
    if x0 !== nothing
        df = df[round.(Int, df.x) .== Int(round(x0)), :]
    end
    if y0 !== nothing
        df = df[round.(Int, df.y) .== Int(round(y0)), :]
    end
    nrow(df) > 0 || error("No rows after filtering for A=$(A),B=$(B),m=$(m),E=$(E), x0=$(x0), y0=$(y0).")

    # Sort by kappa
    kappas = df.kappa
    gaps = df.localiser_gap
    cherns = df.chern_number

    order = sortperm(kappas)
    kappas = kappas[order]; gaps = gaps[order]; cherns = cherns[order]

    p1 = plot(kappas, gaps;
        xscale=:log10, xlabel="kappa", ylabel="min|λ|", lw=2, marker=:circle,
        title="Localiser min|λ| vs kappa (A=$(A),B=$(B),m=$(m),E=$(E)), x0=$(x0), y0=$(y0)", legend=:topleft)

    p2 = plot(kappas, cherns;
        xscale=:log10, xlabel="kappa", ylabel="chern (localiser)", lw=2, marker=:diamond,
        title="Localiser-derived Chern vs kappa", legend=false)#, ylim=(-2.1,2.1))

    plt = plot(p1, p2; layout=(2,1), size=(900,700))
    isdir(dirname(filename)) || mkpath(dirname(filename))
    savefig(plt, filename)
    display(plt)

end


# ## plot for
# A_plt = 1.0
# B_plt = 1.0
# m_plt = -1.0
# E_plt = 0.0

# x0_plt = 15
# y0_plt = 15

# # Example usage (adjust parameters to match the run that produced localiser_df)
# foldername = joinpath("plots", "SL_kappa_line")
# isdir(foldername) || mkpath(foldername)

# kappa_range = (minimum(localiser_df.kappa), maximum(localiser_df.kappa), length(unique(localiser_df.kappa)))

# plt_localiser_kappa_from_df(
#     kap_scan_localiser_df;
#     A=A_plt, B=B_plt, m=m_plt, x0=x0_plt, y0=y0_plt, E=E_plt,
#     filename=joinpath(foldername, "localiser_kappa_line_A$(A_plt)_B$(B_plt)_m$(m_plt)_E$(E_plt)_Lx$(Lx)_Ly$(Ly)_x0$(x0_plt)_y0$(y0_plt)__kapRange$(kappa_range).png")
# )

In [ ]:
## Do out fo bound position search at known kappa and energy.

Avals = [1.0]
Bvals = [1.0]
mvals = [1.0, -1.0]
Lx = 20
Ly = 20
xs = collect(range(-Lx, 2Lx, length=50))
ys = collect(range(-Ly, 2Ly, length=50)) #collect(1:Ly)
Es = [0.0]
kappas = [1e-2] #logrange(1e-4, 1e4, 30)

pos_bounds_localiser_df = compute_spec_loc_df(
    Lx, 
    Ly; 
    Avals=Avals, 
    Bvals=Bvals, 
    mvals=mvals, 
    xs=xs, 
    ys=ys, 
    Es=Es, 
    kappas=kappas, 
    periodic_x=false, 
    periodic_y=false, 
    sparse_output=true,
    kk_tol=1e-8,
    kk_maxiter=1000,
    gap_tol=1e-6
)

println("Computed spectral localiser DataFrame with $(nrow(pos_bounds_localiser_df)) rows and $(ncol(pos_bounds_localiser_df)) columns.")

In [ ]:
## Position scan plot of localiser gap and chern against position at fixed kappa and energy

## This function only plots for the lattice sites (1..Lx, 1..Ly) and ignores out-of-bound positions.
# function plt_localiser_position_heatmaps(
#     localiser_df; 
#     A::Real=1.0, 
#     B::Real=1.0, 
#     m::Real=0.0, 
#     E::Real=0.0, 
#     kappa::Real=1.0, 
#     filename::String="plots/localiser_position_heatmaps.png"
# )
#     subdf = localiser_df[(localiser_df.A .== Float64(A)) .& (localiser_df.B .== Float64(B)) .& (localiser_df.m .== Float64(m)) .& (localiser_df.E .== Float64(E)), :]
#     nrow(subdf) > 0 || error("No rows in localiser_df for A=$(A), B=$(B), m=$(m), E=$(E), kappa=$(kappa)")

#     Lx = Int(subdf.Lx[1]); Ly = Int(subdf.Ly[1])
#     gap_mat = fill(NaN, Lx, Ly)
#     chern_mat = fill(NaN, Lx, Ly)

#     for row in eachrow(subdf)
#         xi = Int(round(row.x)); yi = Int(round(row.y))
#         if 1 <= xi <= Lx && 1 <= yi <= Ly
#             gap_mat[xi, yi] = row.localiser_gap
#             # sig_mat[xi, yi] = row.signature
#             chern_mat[xi, yi] = ismissing(row.chern_number) ? NaN : Float64(row.chern_number)
#         end
#     end

#     # Bulk center marker
#     xc = Int(round(Lx/2)); yc = Int(round(Ly/2))

#     p1 = heatmap(1:Lx, 1:Ly, gap_mat';
#         xlabel="x", ylabel="y",
#         title="min |λ| — A=$(A) B=$(B) m=$(m) E=$(E) kappa=$(kappa)",
#         colorbar_title="min|λ|",
#         aspect_ratio=1)

#     # scatter!(p1, [xc], [yc]; marker=:x, ms=8, mc=:black, label=false)

#     # p2 = heatmap(1:Lx, 1:Ly, sig_mat';
#     #     xlabel="x", ylabel="y",
#     #     title="Localiser signature (npos - nneg)",
#     #     colorbar_title="signature",
#     #     aspect_ratio=1)

#     # scatter!(p2, [xc], [yc]; marker=:x, ms=8, mc=:black, label=false)

#     p3 = heatmap(1:Lx, 1:Ly, chern_mat';
#         xlabel="x", ylabel="y",
#         title="Chern",
#         colorbar_title=L"C",
#         clims=(minimum(chern_mat, dims=1)[1], maximum(chern_mat, dims=1)[1]),
#         aspect_ratio=1)

#     # scatter!(p3, [xc], [yc]; marker=:x, ms=8, mc=:black, label=false)

#     plt = plot(p1, p3; layout=(1,2), size=(1400,420))
#     display(plt)
#     isdir(dirname(filename)) || mkpath(dirname(filename))
#     savefig(plt, filename)
# end

## This function plots for actual positions -- correct
function plt_localiser_position_heatmaps_sparse(
    localiser_df::DataFrame; 
    A::Real=1.0, B::Real=1.0, m::Real=0.0, E::Real=0.0, kappa::Real=1.0,
    filename::String="plots/localiser_position_heatmaps_sparse.png"
)
    
    # Filter by parameters
    subdf = localiser_df[(localiser_df.A .== Float64(A)) .& (localiser_df.B .== Float64(B)) .& 
                         (localiser_df.m .== Float64(m)) .& (localiser_df.E .== Float64(E)) .& (localiser_df.kappa .== Float64(kappa)), :]
    nrow(subdf) > 0 || error("No rows in localiser_df for A=$(A), B=$(B), m=$(m), E=$(E), kappa=$(kappa)")

    # Extract unique x, y positions (as they appear in the data)
    xs = sort(unique(subdf.x))
    ys = sort(unique(subdf.y))
    nx, ny = length(xs), length(ys)
    
    # Create lookup dicts
    x_idx = Dict(xs[i] => i for i in 1:nx)
    y_idx = Dict(ys[i] => i for i in 1:ny)

    # Initialize matrices for the actual data points only
    gap_mat = fill(NaN, ny, nx)  # (y, x) for heatmap orientation
    # sig_mat = fill(NaN, ny, nx)
    chern_mat = fill(NaN, ny, nx)

    for row in eachrow(subdf)
        xi = x_idx[row.x]
        yi = y_idx[row.y]
        gap_mat[yi, xi] = row.localiser_gap
        # sig_mat[yi, xi] = row.signature
        chern_mat[yi, xi] = ismissing(row.chern_number) ? NaN : Float64(row.chern_number)
    end

    p1 = heatmap(xs, ys, gap_mat;
        xlabel="x", ylabel="y",
        title="Localiser min|λ| — A=$(A) B=$(B) m=$(m) E=$(E)",
        colorbar_title="min|λ|",
        aspect_ratio=:auto)

    p3 = heatmap(xs, ys, chern_mat;
        xlabel="x", ylabel="y",
        title="Localiser-derived Chern",
        colorbar_title="chern",
        # clims=(-2, 2),
        aspect_ratio=:auto)

    plt = plot(p1, p3; layout=(1, 2), size=(1400, 420))
    isdir(dirname(filename)) || mkpath(dirname(filename))
    savefig(plt, filename)
    display(plt)
end



## plot for
A_plt = 1.0
B_plt = 1.0
m_plt = 1.0
E_plt = 0.0
kappa_plt = 0.01

# Example usage
foldername = joinpath("plots", "OOB_SL_spatial_scan")
isdir(foldername) || mkpath(foldername)

plt_localiser_position_heatmaps_sparse(
    # pos_bounds_localiser_df; 
    localiser_df;
    A=A_plt, 
    B=B_plt, 
    m=m_plt, 
    E=E_plt, 
    kappa=kappa_plt, 
    filename=joinpath(foldername, "new_localiser_position_heatmaps_m$(m_plt)_B$(B_plt)_A$(A_plt)_E$(E_plt)_kappa$(kappa_plt).png")
)

In [ ]:
## search kappa and enegry space at known position to get heatmap

Avals = [1.0]
Bvals = [1.0]
mvals = [1.0, -1.0]
Lx = 20
Ly = 20
xs = [1.5] #collect(range(-Lx, 2Lx, length=50))
ys = [1.5] #collect(range(-Ly, 2Ly, length=50)) #collect(1:Ly)
Es = collect(range(-6.0, 6.0, length=70))
kappas = logrange(1e-4, 1e2, 50)

# kap_energy_localiser_df = compute_spec_loc_df(
#     Lx, 
#     Ly; 
#     Avals=Avals, 
#     Bvals=Bvals, 
#     mvals=mvals, 
#     xs=xs, 
#     ys=ys, 
#     Es=Es, 
#     kappas=kappas, 
#     periodic_x=false, 
#     periodic_y=false, 
#     sparse_output=true,
#     kk_tol=1e-8,
#     kk_maxiter=1000,
#     gap_tol=1e-6
# )

kap_energy_localiser_df = compute_spec_loc_chern_only_df(
    Lx, 
    Ly; 
    Avals=Avals, 
    Bvals=Bvals, 
    mvals=mvals, 
    xs=xs, 
    ys=ys, 
    Es=Es, 
    kappas=kappas, 
    periodic_x=false, 
    periodic_y=false, 
    sparse_output=true,
    kk_tol=1e-8,
    kk_maxiter=1000,
    gap_tol=1e-6
)

println("Computed spectral localiser DataFrame with $(nrow(kap_energy_localiser_df)) rows and $(ncol(kap_energy_localiser_df)) columns.")

In [ ]:
function heatmap_localiser_kappa_energy(
    localiser_df::DataFrame; 
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma::Real=0.0,
    logscale::Bool=true,
    x0::Union{Nothing,Real}=nothing, 
    y0::Union{Nothing,Real}=nothing,
    filename::String="plots/localiser_kappa_energy_heatmap.png"
)
    # Filter by parameters
    df = localiser_df[(localiser_df.A .== Float64(A)) .& (localiser_df.B .== Float64(B)) .& (localiser_df.m .== Float64(m)) .& (localiser_df.gamma .== Float64(gamma)), :]
    if x0 !== nothing
        df = df[round.(Int, df.x) .== Int(round(x0)), :]
    end
    if y0 !== nothing
        df = df[round.(Int, df.y) .== Int(round(y0)), :]
    end
    nrow(df) > 0 || error("No rows after filtering for A=$(A),B=$(B),m=$(m), x0=$(x0), y0=$(y0).")

    kappas = sort(unique(df.kappa))
    Es = sort(unique(df.E))

    gap_mat = fill(NaN, length(Es), length(kappas))
    chern_mat = fill(NaN, length(Es), length(kappas))

    for row in eachrow(df)
        k_idx = findfirst(==(row.kappa), kappas)
        e_idx = findfirst(==(row.E), Es)
        gap_mat[e_idx, k_idx] = row.localiser_gap
        chern_mat[e_idx, k_idx] = ismissing(row.chern_number) ? NaN : Float64(row.chern_number)
    end

    if logscale
        gap_mat = log10.(gap_mat)
    end

    p1 = heatmap(kappas, Es, gap_mat;
        xlabel="kappa", ylabel="E",
        title="Localiser min|λ| — A=$(A) B=$(B) m=$(m) g=$(gamma)",
        xscale=:log10,
        colorbar_title="min|λ|"
    )

    p2 = heatmap(kappas, Es, chern_mat;
        xlabel="kappa", ylabel="E",
        title="Localiser-derived Chern",
        xscale=:log10,
        colorbar_title="chern"
    )

    plt = plot(p1, p2; layout=(1, 2), size=(1000, 600))
    display(plt)
    isdir(dirname(filename)) || mkpath(dirname(filename))
    savefig(plt, filename)
end

function heatmap_localiser_kappa_energy_chern_only(
    localiser_df::DataFrame; 
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    x0::Union{Nothing,Real}=nothing, 
    y0::Union{Nothing,Real}=nothing,
    filename::String="plots/localiser_kappa_energy_heatmap.png"
)
    # Filter by parameters
    df = localiser_df[(localiser_df.A .== Float64(A)) .& (localiser_df.B .== Float64(B)) .& (localiser_df.m .== Float64(m)), :]
    if x0 !== nothing
        df = df[round.(Int, df.x) .== Int(round(x0)), :]
    end
    if y0 !== nothing
        df = df[round.(Int, df.y) .== Int(round(y0)), :]
    end
    nrow(df) > 0 || error("No rows after filtering for A=$(A),B=$(B),m=$(m), x0=$(x0), y0=$(y0).")

    kappas = sort(unique(df.kappa))
    Es = sort(unique(df.E))

    # gap_mat = fill(NaN, length(Es), length(kappas))
    chern_mat = fill(NaN, length(Es), length(kappas))

    for row in eachrow(df)
        k_idx = findfirst(==(row.kappa), kappas)
        e_idx = findfirst(==(row.E), Es)
        # gap_mat[e_idx, k_idx] = row.localiser_gap
        chern_mat[e_idx, k_idx] = ismissing(row.chern_number) ? NaN : Float64(row.chern_number)
    end

    # p2 = heatmap(kappas, Es, chern_mat;
    #     xlabel="kappa", ylabel="E",
    #     title="Localiser-derived Chern",
    #     xscale=:log10,
    #     colorbar_title="chern"
    # )
    # Get unique Chern values in the data (excluding NaN)
    chern_unique = sort(unique(chern_mat[.!isnan.(chern_mat)]))

    p2 = heatmap(kappas, Es, chern_mat;
        xlabel="kappa", ylabel="E",
        title="Localiser-derived Chern",
        xscale=:log10,
        colorbar_title="chern",
        # Set color limits to the range of Chern values
        clims=(minimum(chern_unique) - 0.5, maximum(chern_unique) + 0.5),
        # Use a discrete palette (e.g., :Set1_9 for up to 9 discrete colors)
        palette=:Set1_9,
        # Set colorbar ticks to match discrete values
        colorbar_ticks=(chern_unique, string.(chern_unique))
    )

    plt = plot(p2; layout=(1, 1), size=(1000, 600))
    display(plt)
    isdir(dirname(filename)) || mkpath(dirname(filename))
    savefig(plt, filename)
end


# ## plot for
# A_plt = 1.0
# B_plt = 1.0
# m_plt = -1.0
# x0_plt = 1.5
# y0_plt = 1.5
# Es_length = length(Es)
# kappa_length = length(kappas)

# # Example usage
# foldername = joinpath("plots", "SL_kap_energy_scan")
# isdir(foldername) || mkpath(foldername)

# # heatmap_localiser_kappa_energy(
# #     kap_energy_localiser_df;
# #     A=A_plt, 
# #     B=B_plt, 
# #     m=m_plt, 
# #     x0=x0_plt,
# #     y0=y0_plt,
# #     filename=joinpath(foldername, "localiser_kappa_energy_heatmap_x0$(x0_plt)_y0$(y0_plt)_A$(A_plt)_B$(B_plt)_m$(m_plt).png")
# # )

# heatmap_localiser_kappa_energy_chern_only(
#     kap_energy_localiser_df;
#     A=A_plt, 
#     B=B_plt, 
#     m=m_plt, 
#     x0=x0_plt,
#     y0=y0_plt,
#     filename=joinpath(foldername, "localiser_kappa_energy_heatmap_x0$(x0_plt)_y0$(y0_plt)_A$(A_plt)_B$(B_plt)_m$(m_plt)_nkappas$(kappa_length)_nEs$(Es_length).png")
# )

In [ ]:
## Repeat the validation for smaller/larger system sizes for stability

In [ ]:
## Direct comparison of the localiser Chern number with the analytic and numerical Chern numbers for the same parameters

##### Val 2: Slow

In [ ]:
## Do kappa search at known parameters, positions and energy

Avals = [1.0]
Bvals = [1.0]
mvals = [1.0, -1.0]
Lx = 20
Ly = 20
xs = [10.0] # collect(range(-Lx, 2Lx, length=50))
ys = [10.0] # collect(range(-Ly, 2Ly, length=50)) #collect(1:Ly)
Es = [0.0]
kappas = logrange(1e-4, 1e4, 30)

slow_localiser_df_kap_Scan = slow_compute_spectral_localiser_df(
    Lx, 
    Ly; 
    Avals=Avals, 
    Bvals=Bvals, 
    mvals=mvals, 
    xs=xs, 
    ys=ys, 
    Es=Es, 
    kappas=kappas, 
    periodic_x=false, 
    periodic_y=false, 
    sparse_output=true
)

println("Computed spectral localiser DataFrame with $(nrow(slow_localiser_df_kap_Scan)) rows and $(ncol(slow_localiser_df_kap_Scan)) columns.")

In [ ]:
## plot for
A_plt = 1.0
B_plt = 1.0
m_plt = -1.0
E_plt = 0.0

x0_plt = 10.0
y0_plt = 10.0

# Example usage (adjust parameters to match the run that produced localiser_df)
foldername = joinpath("plots", "SL_kappa_line_slow")
isdir(foldername) || mkpath(foldername)

kappa_range = (minimum(slow_localiser_df_kap_Scan.kappa), maximum(slow_localiser_df_kap_Scan.kappa), length(unique(slow_localiser_df_kap_Scan.kappa)))

plt_localiser_kappa_from_df(
    slow_localiser_df_kap_Scan;
    A=A_plt, B=B_plt, m=m_plt, x0=x0_plt, y0=y0_plt, E=E_plt,
    filename=joinpath(foldername, "localiser_kappa_line_A$(A_plt)_B$(B_plt)_m$(m_plt)_E$(E_plt)_Lx$(Lx)_Ly$(Ly)_x0$(x0_plt)_y0$(y0_plt)__kapRange$(kappa_range).png")
)

In [ ]:
## search kappa and energy space at known position to get heatmap

Avals = [1.0]
Bvals = [1.0]
mvals = [1.0, -1.0]
Lx = 20
Ly = 20
xs = [10.0] #collect(range(-Lx, 2Lx, length=50))
ys = [10.0] #collect(range(-Ly, 2Ly, length=50)) #collect(1:Ly)
Es = collect(range(-4.0, 4.0, length=20))
kappas = logrange(1e-2, 1e1, 15)

slow_localiser_df_energy_kap_Scan = slow_compute_spectral_localiser_df(
    Lx, 
    Ly; 
    Avals=Avals, 
    Bvals=Bvals, 
    mvals=mvals, 
    xs=xs, 
    ys=ys, 
    Es=Es, 
    kappas=kappas, 
    periodic_x=false, 
    periodic_y=false, 
    sparse_output=true
)

println("Computed spectral localiser DataFrame with $(nrow(slow_localiser_df_energy_kap_Scan)) rows and $(ncol(slow_localiser_df_energy_kap_Scan)) columns.")

In [ ]:
## plot for
A_plt = 1.0
B_plt = 1.0
m_plt = -1.0
x0_plt = 10.0
y0_plt = 10.0
Es_length = length(Es)
kappa_length = length(kappas)

# Example usage
foldername = joinpath("plots", "SL_kap_energy_scan_slow")
isdir(foldername) || mkpath(foldername)

heatmap_localiser_kappa_energy(
    slow_localiser_df_energy_kap_Scan;
    A=A_plt, 
    B=B_plt, 
    m=m_plt, 
    x0=x0_plt,
    y0=y0_plt,
    filename=joinpath(foldername, "localiser_kappa_energy_heatmap_x0$(x0_plt)_y0$(y0_plt)_A$(A_plt)_B$(B_plt)_m$(m_plt).png")
)

heatmap_localiser_kappa_energy_chern_only(
    slow_localiser_df_energy_kap_Scan;
    A=A_plt, 
    B=B_plt, 
    m=m_plt, 
    x0=x0_plt,
    y0=y0_plt,
    filename=joinpath(foldername, "localiser_kappa_energy_heatmap_x0$(x0_plt)_y0$(y0_plt)_A$(A_plt)_B$(B_plt)_m$(m_plt)_nkappas$(kappa_length)_nEs$(Es_length).png")
)

## Bandgap Manipulation

The aim is to add terms to the Hamiltonian which will distort the bandsctructure into giving an indirect bandgap. 

First we will add terms directly to the momentum space Hamiltonian as dispersive terms on the identitiy operator.

$$H(\vec{k}) = g(\vec{k})\mathbb{I} + H_0(\vec{k})$$

e.g.
1) $g_s(\vec{k}) = \gamma_s (\cos{(k_x)} + \cos{(k_y)})$
2) $g_t(\vec{k}) = \gamma_t \sin{(k_x)}$ 



In [ ]:
function symmetric_perturbed_qwz_hamiltonian(
    kx::Real,
    ky::Real;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma_s::Real=0.0
)::Tuple{Matrix{ComplexF64}, Tuple{Float64, Float64}}

    dx::Float64 = Float64(A) * sin(Float64(kx))
    dy::Float64 = Float64(A) * sin(Float64(ky))
    dz::Float64 = Float64(m) + Float64(B) * (2.0 - cos(Float64(kx)) - cos(Float64(ky)))
    d_I::Float64 = Float64(gamma_s) * (cos(Float64(kx)) + cos(Float64(ky)))
     
    H::Matrix{ComplexF64} = ComplexF64.((dx * sigma_x + dy * sigma_y + dz * sigma_z) + d_I * I)
    d_norm = sqrt(dx * dx + dy * dy + dz * dz)
    es = (d_I + d_norm, d_I - d_norm)
    # e::Float64 = sqrt(dx * dx + dy * dy + dz * dz + d_I * d_I)
    # es = (e, -e)
    
    return H, es
end

function tilt_perturbed_qwz_hamiltonian(
    kx::Real,
    ky::Real;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma_t::Real=0.0
)::Tuple{Matrix{ComplexF64}, Tuple{Float64, Float64}}

    dx::Float64 = Float64(A) * sin(Float64(kx))
    dy::Float64 = Float64(A) * sin(Float64(ky))
    dz::Float64 = Float64(m) + Float64(B) * (2.0 - cos(Float64(kx)) - cos(Float64(ky)))
    d_I::Float64 = Float64(gamma_t) * sin(Float64(kx))
     
    H::Matrix{ComplexF64} = ComplexF64.((dx * sigma_x + dy * sigma_y + dz * sigma_z) + d_I * I)
    d_norm = sqrt(dx * dx + dy * dy + dz * dz)
    es = (d_I + d_norm, d_I - d_norm)
    # e::Float64 = sqrt(dx * dx + dy * dy + dz * dz + d_I * d_I)
    # es = (e, -e)
    
    return H, es
end

# Wide-format table: one row per (A, B, m, kx, ky), with both bands in separate columns.
function build_perturbed_evals_df(
    Avals::AbstractVector{<:Real},
    Bvals::AbstractVector{<:Real},
    mvals::AbstractVector{<:Real},
    gammas::AbstractVector{<:Real},
    perturbation_type::Symbol=:symmetric; # :symmetric, :tilt, :linear
    nk::Int=51
)::DataFrame

    ks::Vector{Float64} = collect(range(-pi, pi; length=nk))[1:end-1]  # avoid duplicate endpoint at +pi
    nrows::Int = length(Avals) * length(Bvals) * length(mvals) * length(gammas) * length(ks)^2

    Acol = Vector{Float64}(undef, nrows)
    Bcol = Vector{Float64}(undef, nrows)
    mcol = Vector{Float64}(undef, nrows)
    gammacol = Vector{Float64}(undef, nrows)
    kxcol = Vector{Float64}(undef, nrows)
    kycol = Vector{Float64}(undef, nrows)
    Eminus_col = Vector{Float64}(undef, nrows)
    Eplus_col = Vector{Float64}(undef, nrows)

    idx::Int = 1
    total_iter = length(Avals) * length(Bvals) * length(mvals) * length(gammas) * length(ks)^2
    @showprogress for A in Avals, B in Bvals, m in mvals, gamma in gammas, kx in ks, ky in ks
        if perturbation_type == :symmetric
            _, es = symmetric_perturbed_qwz_hamiltonian(kx, ky; A=A, B=B, m=m, gamma_s=gamma)
        elseif perturbation_type == :tilt
            _, es = tilt_perturbed_qwz_hamiltonian(kx, ky; A=A, B=B, m=m, gamma_t=gamma)
        elseif perturbation_type == :linear
            _, es = linear_perturbed_qwz_hamiltonian(kx, ky; A=A, B=B, m=m, gamma_t=gamma)
        else
            error("Unknown perturbation type: $perturbation_type")
        end
        @inbounds begin
            Acol[idx] = Float64(A); Bcol[idx] = Float64(B); mcol[idx] = Float64(m); gammacol[idx] = Float64(gamma)
            kxcol[idx] = kx; kycol[idx] = ky
            Eminus_col[idx] = es[2]; Eplus_col[idx] = es[1]
            idx += 1
        end
    end

    return DataFrame(
        A=Acol,
        B=Bcol,
        m=mcol,
        gamma=gammacol,
        kx=kxcol,
        ky=kycol,
        E_minus=Eminus_col,
        E_plus=Eplus_col
    )
end

In [ ]:
Avals = [1.0, 2.0]#, 2.0, 5.0]
Bvals = collect(1.0:0.1:2.0) #[1.0, 2.0]
mvals = collect(-4.0:0.5:1.0) #
gammavals = collect(0.0:0.5:4.0) #[0.0, 0.5, 1.0] #
nk = 51
perturbation_type = :symmetric # symmetric or :tilt

pert_qwz_df = build_perturbed_evals_df(Avals, Bvals, mvals, gammavals, perturbation_type; nk=nk)
println("Built DataFrame with $(nrow(pert_qwz_df)) rows and $(ncol(pert_qwz_df)) columns.")

In [ ]:
# plot for
A = 1.0
B = 1.0
m = -2.0
gamma = 1.0
perturbation_type = :symmetric # or :tilt

foldername = "plots/perturbed_$(perturbation_type)_combined_bs"
isdir(foldername) || mkdir(foldername)
filename = joinpath(foldername, "perturbed_$(perturbation_type)_qwz_bandstructure_combined_m$(m)_A$(A)_B$(B)_gamma$(gamma).png")

plt_qwz_bandstructure_combined(
    pert_qwz_df; 
    filename=filename, 
    alpha=0.75,
    m=m, A=A, B=B, gamma=gamma)

In [ ]:
function plt_qwz_bandstructure_path(
    df::DataFrame;
    atol::Real=1e-2,
    filename::String="qwz_bandstructure_path.png",
    fixed_variables...
)
    # 1. Verify required columns exist
    required_cols = ["kx", "ky", "E_minus", "E_plus"]
    col_names_str = string.(names(df))
    for c in required_cols
        c in col_names_str || error("Missing required column: $c")
    end

    # 2. Filter dataframe using fixed_variables
    subdf = df
    for (k, v) in fixed_variables
        string(k) in col_names_str || error("Filter key $(k) is not a DataFrame column.")
        col = subdf[!, k]
        if v isa Real && eltype(col) <: Real
            mask = abs.(Float64.(col) .- Float64(v)) .<= Float64(atol)
            subdf = subdf[mask, :]
        else
            mask = col .== v
            subdf = subdf[mask, :]
        end
    end

    nrow(subdf) > 0 || error("No rows matched the requested fixed variables. Try adjusting atol.")

    # 3. Determine Brillouin zone coordinates bounds
    min_k = minimum(subdf.kx)
    max_k = maximum(subdf.kx)
    zero_k = subdf.kx[argmin(abs.(subdf.kx))]

    # 4. Extract path segments: Γ -> X -> M -> Γ (with proper parentheses around comparisons)
    # Segment 1: Γ(0,0) to X(max_k, 0)
    seg1_mask = (abs.(subdf.ky .- zero_k) .<= atol) .& (subdf.kx .>= zero_k) .& (subdf.kx .<= max_k)
    seg1 = sort(subdf[seg1_mask, :], :kx)

    # Segment 2: X(max_k, 0) to M(max_k, max_k)
    seg2_mask = (abs.(subdf.kx .- max_k) .<= atol) .& (subdf.ky .>= zero_k) .& (subdf.ky .<= max_k)
    seg2 = sort(subdf[seg2_mask, :], :ky)

    # Segment 3: M(max_k, max_k) to Γ(0,0)
    seg3_mask = (abs.(subdf.kx .- subdf.ky) .<= atol) .& (subdf.kx .>= zero_k) .& (subdf.kx .<= max_k)
    seg3 = sort(subdf[seg3_mask, :], :kx, rev=true)

    # Combine segments while dropping duplicate junction points
    n1 = nrow(seg1)
    n2 = nrow(seg2) - 1
    n3 = nrow(seg3) - 1
    
    (n1 > 0 && n2 > 0 && n3 > 0) || error("Could not resolve full high-symmetry path from the grid data. Check grid resolution.")

    path_rows = vcat(seg1, seg2[2:end, :], seg3[2:end, :])

    # 5. Compute cumulative distance along the path for the x-axis
    s = zeros(Float64, nrow(path_rows))
    for i in 2:nrow(path_rows)
        dkx = path_rows.kx[i] - path_rows.kx[i-1]
        dky = path_rows.ky[i] - path_rows.ky[i-1]
        s[i] = s[i-1] + sqrt(dkx^2 + dky^2)
    end

    # Define high-symmetry tick positions
    tick_vals = [s[1], s[n1], s[n1 + n2], s[end]]
    tick_labels = ["Γ", "X", "M", "Γ"]

    # 6. Plot the bands
    plt = plot(s, path_rows.E_minus; label="E_minus", color=:blue, linewidth=2)
    plot!(plt, s, path_rows.E_plus; label="E_plus", color=:red, linewidth=2)

    # Add vertical dashed lines at high-symmetry points
    for t in tick_vals[2:end-1]
        vline!(plt, [t]; color=:gray, linestyle=:dash, label="")
    end

    # Formatting options
    xlabel!("Path through Brillouin Zone")
    ylabel!("Energy (E)")
    xticks!(tick_vals, tick_labels)
    xlims!(s[1], s[end])

    label_text = isempty(fixed_variables) ? "All rows" : join(["$(k)=$(v)" for (k, v) in fixed_variables], ", ")
    plot!(plt; title="QWZ Band Structure (Path) \n" * label_text, size=(900, 500))
    
    display(plt)
    savefig(plt, filename)
end

In [ ]:
# plot for
A = 1.0
B = 1.0
m = -1.0
gamma = 2.0
perturbation_type = :symmetric # or :tilt

foldername = "plots/perturbed_$(perturbation_type)_combined_hs_bs"
isdir(foldername) || mkdir(foldername)
filename = joinpath(foldername, "perturbed_$(perturbation_type)_qwz_bandstructure_combined_m$(m)_A$(A)_B$(B)_gamma$(gamma).png")

plt_qwz_bandstructure_path(
    pert_qwz_df; 
    filename=filename, 
    m=m, A=A, B=B, gamma=gamma)

In [ ]:
# 1. Unified Block Generator
function real_space_perturbed_qwz_blocks(; 
    A::Real=1.0, 
    B::Real=1.0, 
    m::Real=0.0, 
    gamma::Real=0.0, 
    perturbation_type::Symbol=:none
)
    # Base QWZ blocks
    onsite = ComplexF64.((m + 2.0 * B) * sigma_z)
    tx = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_x)
    ty = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_y)

    # Apply perturbations
    if perturbation_type == :symmetric
        # gamma * (cos(kx) + cos(ky)) * I
        tx += ComplexF64.(0.5 * gamma * identity)
        ty += ComplexF64.(0.5 * gamma * identity)
    elseif perturbation_type == :tilt
        # gamma * sin(kx) * I
        tx += ComplexF64.(0.5im * gamma * identity)
        # ty is unperturbed by the 1D tilt
    elseif perturbation_type == :none
        # Base QWZ, do nothing
    else
        error("Unknown perturbation type: $perturbation_type")
    end

    return onsite, tx, ty
end

# 2. Updated Real-Space Hamiltonian Builder
function real_space_perturbed_hamiltonian_qwz(
    Lx::Int,
    Ly::Int;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma::Real=0.0,
    perturbation_type::Symbol=:none,
    periodic_x::Bool=false,
    periodic_y::Bool=false,
    sparse_output::Bool=true
)
    Lx > 0 || error("Lx must be positive.")
    Ly > 0 || error("Ly must be positive.")

    # Get the perturbed blocks
    onsite, tx, ty = real_space_perturbed_qwz_blocks(; A=A, B=B, m=m, gamma=gamma, perturbation_type=perturbation_type)
    
    nsites = Lx * Ly
    dim = 2 * nsites
    H = sparse_output ? spzeros(ComplexF64, dim, dim) : zeros(ComplexF64, dim, dim)

    function add_block!(mat, row_site::Tuple{Int, Int}, col_site::Tuple{Int, Int}, block::AbstractMatrix{<:Number})
        (xr, yr) = row_site
        (xc, yc) = col_site
        row_base = site_index_qwz(xr, yr, 1, Lx, Ly)
        col_base = site_index_qwz(xc, yc, 1, Lx, Ly)
        @inbounds for a in 0:1, b in 0:1
            mat[row_base + a, col_base + b] += ComplexF64(block[a + 1, b + 1])
        end
        return nothing
    end

    for y in 1:Ly, x in 1:Lx
        add_block!(H, (x, y), (x, y), onsite)

        if x < Lx
            add_block!(H, (x + 1, y), (x, y), tx)
            add_block!(H, (x, y), (x + 1, y), tx')
        elseif periodic_x
            add_block!(H, (1, y), (x, y), tx)
            add_block!(H, (x, y), (1, y), tx')
        end

        if y < Ly
            add_block!(H, (x, y + 1), (x, y), ty)
            add_block!(H, (x, y), (x, y + 1), ty')
        elseif periodic_y
            add_block!(H, (x, 1), (x, y), ty)
            add_block!(H, (x, y), (x, 1), ty')
        end
    end

    return H
end

# 3. Updated Spectral Localizer Wrapper
function slow_compute_perturbed_spectral_localiser_df(
    Lx::Int, 
    Ly::Int; 
    Avals::AbstractVector{<:Real} = [1.0], 
    Bvals::AbstractVector{<:Real} = [1.0], 
    mvals::AbstractVector{<:Real} = [0.0],
    gammas::AbstractVector{<:Real} = [0.0],
    perturbation_type::Symbol = :none,
    xs::AbstractVector = collect(1:Lx), 
    ys::AbstractVector = collect(1:Ly), 
    Es::AbstractVector = [0.0],
    kappas::AbstractVector{<:Real} = [1.0], 
    periodic_x::Bool=false, 
    periodic_y::Bool=false, 
    sparse_output::Bool=false
)::DataFrame

    rows = Vector{NamedTuple{(:A, :B, :m, :gamma, :Lx, :Ly, :x, :y, :E, :kappa, :localiser_gap, :signature, :chern_number),
                  Tuple{Float64,Float64,Float64,Float64,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Int64,Union{Missing, Int8}}}}()

    # Pre-build position operators (they don't depend on Hamiltonian parameters)
    X, Y = build_position_operators(Lx, Ly)

    @showprogress for A in Avals, B in Bvals, m in mvals, gamma in gammas, x0 in xs, y0 in ys, E in Es, kappa in kappas
        
        # Build perturbed H for this parameter set
        H = real_space_perturbed_hamiltonian_qwz(Lx, Ly; 
                A=A, B=B, m=m, gamma=gamma, 
                perturbation_type=perturbation_type, 
                periodic_x=periodic_x, periodic_y=periodic_y, sparse_output=sparse_output)
        
        sig, minabs = spectral_localiser_signature(H, X, Y, x0, y0, E; kappa=kappa)
        chern = chern_from_localiser_signature(sig, minabs; zero_tol=1e-12, chern_tol=1e-6)
        
        push!(rows, (A=Float64(A), B=Float64(B), m=Float64(m), gamma=Float64(gamma), Lx=Lx, Ly=Ly, 
                     x=Float64(x0), y=Float64(y0), E=Float64(E), kappa=Float64(kappa), 
                     localiser_gap=Float64(minabs), signature=Int64(sig), chern_number=chern))
    end

    return DataFrame(rows)
end

In [ ]:
## search kappa and energy space at known position to get heatmap

Avals = [1.0]
Bvals = [1.0]
mvals = [1.0, -1.0]
gammavals = [0.0, 1.0, 2.0, 3.0]
Lx = 10
Ly = 10
xs = [5.0] #collect(range(-Lx, 2Lx, length=50))
ys = [5.0] #collect(range(-Ly, 2Ly, length=50)) #collect(1:Ly)
Es = collect(range(-12.0, 12.0, length=60))
kappas = logrange(1e-3, 5e1, 25)

slow_perturbed_localiser_df_energy_kap_Scan = slow_compute_perturbed_spectral_localiser_df(
    perturbation_type=:symmetric,
    Lx, 
    Ly; 
    Avals=Avals, 
    Bvals=Bvals, 
    mvals=mvals, 
    gammas=gammavals,
    xs=xs, 
    ys=ys, 
    Es=Es, 
    kappas=kappas, 
    periodic_x=false, 
    periodic_y=false, 
    sparse_output=true
)

println("Computed spectral localiser DataFrame with $(nrow(slow_perturbed_localiser_df_energy_kap_Scan)) rows and $(ncol(slow_perturbed_localiser_df_energy_kap_Scan)) columns.")

In [ ]:
## save the DataFrame to a JLD2 file for later analysis
output_csv_filename = "data/perturbed_spectral_localiser_df_energy_kap_scan_Lx20_ly20.jld2"
isdir(dirname(output_csv_filename)) || mkpath(dirname(output_csv_filename))

@save output_csv_filename slow_localiser_df_energy_kap_Scan

In [ ]:
## plot for
A_plt = 1.0
B_plt = 1.0
# m_plt = -1.0
x0_plt = 10.0
y0_plt = 10.0
Es_length = length(Es)
kappa_length = length(kappas)

for m in mvals, gamma in gammavals
    foldername = joinpath("plots", "SL_perturbed_symmetric_kap_energy_scan_slow")
    isdir(foldername) || mkpath(foldername)

    heatmap_localiser_kappa_energy(
        slow_localiser_df_energy_kap_Scan[(slow_localiser_df_energy_kap_Scan.m .== m) .& (slow_localiser_df_energy_kap_Scan.gamma .== gamma), :];
        A=A_plt, 
        B=B_plt, 
        m=m, 
        gamma=gamma,
        x0=x0_plt,
        y0=y0_plt,
        filename=joinpath(foldername, "localiser_kappa_energy_heatmap_x0$(x0_plt)_y0$(y0_plt)_A$(A_plt)_B$(B_plt)_m$(m)_gamma$(gamma).png")
    )
end


In [ ]:
## search energy and space at good kappa

Avals = [1.0]
Bvals = [1.0]
mvals = [1.0, -1.0]
gammavals = [0.0, 2.0]
Lx = 10
Ly = 10
xs = collect(range(-1, Lx+1, length=30))
ys = collect(range(-1, Ly+1, length=30)) #collect(1:Ly)
Es = collect(range(-10.0, 10.0, length=50))
kappas = [2e-1]#logrange(1e-3, 5e1, 25)

slow_perturbed_localiser_df_energy_space_scan = slow_compute_perturbed_spectral_localiser_df(
    perturbation_type=:symmetric,
    Lx, 
    Ly; 
    Avals=Avals, 
    Bvals=Bvals, 
    mvals=mvals, 
    gammas=gammavals,
    xs=xs, 
    ys=ys, 
    Es=Es, 
    kappas=kappas, 
    periodic_x=false, 
    periodic_y=false, 
    sparse_output=true
)

println("Computed spectral localiser DataFrame with $(nrow(slow_perturbed_localiser_df_energy_space_scan)) rows and $(ncol(slow_perturbed_localiser_df_energy_space_scan)) columns.")

### Ribbon Geometry

In [ ]:
using LinearAlgebra
using Printf
using Plots
using DataFrames
using SparseArrays
using ProgressMeter
using Statistics
using LaTeXStrings
using JLD2: @save, @load
using Base.Threads

println("Julia Threads: ", Threads.nthreads())
println("BLAS Threads:  ", LinearAlgebra.BLAS.get_num_threads())


## Pauli matrices
sigma_x = [0 1; 1 0]
sigma_y = [0 -im; im 0]
sigma_z = [1 0; 0 -1]
identity = [1 0; 0 1]

# 1. Unified Block Generator
function real_space_perturbed_qwz_blocks(; 
    A::Real=1.0, 
    B::Real=1.0, 
    m::Real=0.0, 
    gamma::Real=0.0, 
    perturbation_type::Symbol=:none
)
    # Base QWZ blocks
    onsite = ComplexF64.((m + 2.0 * B) * sigma_z)
    tx = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_x)
    ty = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_y)

    # Apply perturbations
    if perturbation_type == :symmetric
        # gamma * (cos(kx) + cos(ky)) * I
        tx += ComplexF64.(0.5 * gamma * identity)
        ty += ComplexF64.(0.5 * gamma * identity)
    elseif perturbation_type == :tilt
        # gamma * sin(kx) * I
        tx += ComplexF64.(0.5im * gamma * identity)
        # ty is unperturbed by the 1D tilt
    elseif perturbation_type == :none
        # Base QWZ, do nothing
    else
        error("Unknown perturbation type: $perturbation_type")
    end

    return onsite, tx, ty
end

# Map a site and orbital to a matrix index in the 2LxLy basis.
site_index_qwz(x::Int, y::Int, orb::Int, Lx::Int, Ly::Int) = 2 * ((y - 1) * Lx + (x - 1)) + orb


# 2. Updated Real-Space Hamiltonian Builder
function real_space_perturbed_hamiltonian_qwz(
    Lx::Int,
    Ly::Int;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma::Real=0.0,
    perturbation_type::Symbol=:none,
    periodic_x::Bool=false,
    periodic_y::Bool=false,
    sparse_output::Bool=true
)
    Lx > 0 || error("Lx must be positive.")
    Ly > 0 || error("Ly must be positive.")

    # Get the perturbed blocks
    onsite, tx, ty = real_space_perturbed_qwz_blocks(; A=A, B=B, m=m, gamma=gamma, perturbation_type=perturbation_type)
    
    nsites = Lx * Ly
    dim = 2 * nsites
    H = sparse_output ? spzeros(ComplexF64, dim, dim) : zeros(ComplexF64, dim, dim)

    function add_block!(mat, row_site::Tuple{Int, Int}, col_site::Tuple{Int, Int}, block::AbstractMatrix{<:Number})
        (xr, yr) = row_site
        (xc, yc) = col_site
        row_base = site_index_qwz(xr, yr, 1, Lx, Ly)
        col_base = site_index_qwz(xc, yc, 1, Lx, Ly)
        @inbounds for a in 0:1, b in 0:1
            mat[row_base + a, col_base + b] += ComplexF64(block[a + 1, b + 1])
        end
        return nothing
    end

    for y in 1:Ly, x in 1:Lx
        add_block!(H, (x, y), (x, y), onsite)

        if x < Lx
            add_block!(H, (x + 1, y), (x, y), tx)
            add_block!(H, (x, y), (x + 1, y), tx')
        elseif periodic_x
            add_block!(H, (1, y), (x, y), tx)
            add_block!(H, (x, y), (1, y), tx')
        end

        if y < Ly
            add_block!(H, (x, y + 1), (x, y), ty)
            add_block!(H, (x, y), (x, y + 1), ty')
        elseif periodic_y
            add_block!(H, (x, 1), (x, y), ty)
            add_block!(H, (x, y), (x, 1), ty')
        end
    end

    return H
end

function k_space_perturbed_qwz_hamiltonian(
    kx::Real,
    ky::Real;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma::Real=0.0,
    perturbation_type::Symbol=:none
)::Matrix{ComplexF64}

    # Base QWZ Hamiltonian in k-space
    H_k = (m + 2B - B * (cos(kx) + cos(ky))) * sigma_z +
          A * sin(kx) * sigma_x +
          A * sin(ky) * sigma_y

    # Apply perturbations
    if perturbation_type == :symmetric
        H_k += gamma * (cos(kx) + cos(ky)) * identity
    elseif perturbation_type == :tilt
        H_k += gamma * sin(kx) * identity
    elseif perturbation_type == :none
        # Do nothing
    else
        error("Unknown perturbation type: $perturbation_type")
    end

    return ComplexF64.(H_k)
end

## automatically perturb in both x and y 
function ribbon_perturbed_hamiltonian_qwz(
    ky::Real,
    Lx::Int;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma::Real=0.0,
    perturbation_type::Symbol=:none
)::Hermitian{ComplexF64, Matrix{ComplexF64}}

    # 1. Extract the x-direction hopping block from your existing generator
    _, tx, _ = real_space_perturbed_qwz_blocks(; 
        A=A, B=B, m=m, gamma=gamma, perturbation_type=perturbation_type
    )

    # 2. Construct the momentum-dependent on-site block for this specific ky
    onsite_ky = (m + 2.0 * B - B * cos(ky)) * sigma_z + A * sin(ky) * sigma_y
    
    if perturbation_type == :symmetric
        onsite_ky += gamma * cos(ky) * identity
    elseif perturbation_type == :tilt
        # Tilt is purely a sin(kx) term, so it adds 0 to the ky on-site block
    elseif perturbation_type != :none
        error("Unknown perturbation type: $perturbation_type")
    end

    # 3. Build the 2Lx x 2Lx block tridiagonal Hamiltonian
    dim = 2 * Lx
    H = zeros(ComplexF64, dim, dim)

    for x in 1:Lx
        row_col_idx = (2x - 1):(2x)
        
        # Add on-site terms
        H[row_col_idx, row_col_idx] = onsite_ky
        
        # Add hopping terms (if not at the right boundary)
        if x < Lx
            next_idx = (2x + 1):(2x + 2)
            H[row_col_idx, next_idx] = tx
            H[next_idx, row_col_idx] = tx'
        end
    end

    # Wrap in Hermitian to ensure numerical stability and fast diagonalization
    return Hermitian(H)
end

## choose which directions to perturb in independently
function ribbon_perturbed_hamiltonian_qwz(
    ky::Real,
    Lx::Int;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma::Real=0.0,
    perturbation_type::Symbol=:none,
    perturb_x::Bool=false, # standard don't perturn the OBC direction
    perturb_y::Bool=true
)
    Lx > 0 || error("Lx must be positive.")

    # 1. Construct x-hopping block (tx)
    # Base QWZ hopping along x
    tx = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_x)

    # Apply x-perturbation only if perturb_x is true
    if perturb_x
        if perturbation_type == :symmetric
            # gamma * cos(kx) -> hopping term 0.5 * gamma * identity
            tx += ComplexF64.(0.5 * gamma * identity)
        elseif perturbation_type == :tilt
            # gamma * sin(kx) -> hopping term 0.5im * gamma * identity
            tx += ComplexF64.(0.5im * gamma * identity)
        end
    end

    # 2. Construct ky-dependent on-site block
    # Base QWZ on-site term for fixed ky
    onsite_ky = (m + 2.0 * B - B * cos(ky)) * sigma_z + A * sin(ky) * sigma_y

    # Apply y-perturbation only if perturb_y is true
    if perturb_y
        if perturbation_type == :symmetric
            # gamma * cos(ky) term
            onsite_ky += gamma * cos(ky) * identity
        elseif perturbation_type == :tilt
            # Tilt is sin(kx), has no ky component
        end
    end

    # 3. Assemble 2Lx x 2Lx block tridiagonal ribbon matrix
    dim = 2 * Lx
    H = zeros(ComplexF64, dim, dim)

    for x in 1:Lx
        idx = (2x - 1):(2x)
        H[idx, idx] = onsite_ky
        
        if x < Lx
            next_idx = (2x + 1):(2x + 2)
            H[idx, next_idx] = tx
            H[next_idx, idx] = tx'
        end
    end

    return Hermitian(H)
end

function compute_ribbon_spectrum(
    Lx::Int; 
    N_ky::Int=101, 
    kwargs...
)
    ky_vals = range(-π, π, length=N_ky)
    energies = zeros(Float64, 2 * Lx, N_ky)
    
    for (i, ky) in enumerate(ky_vals)
        H_ribbon = ribbon_perturbed_hamiltonian_qwz(ky, Lx; kwargs...)
        energies[:, i] = eigvals(H_ribbon)
    end
    
    return ky_vals, energies
end

# Example usage to plot:
Lx = 30
A = 1.0
B = 1.0
m = -1.0
gammas = [0.0, 0.5, 1.0, 2.0]
perturbation_type = :symmetric
perturb_x = false
perturb_y = true

for gamma in gammas
    ky_vals, evals = compute_ribbon_spectrum(
        Lx,
        A=A,
        B=B,
        m=m, 
        gamma=gamma, 
        perturbation_type=perturbation_type,
        perturb_x=perturb_x,
        perturb_y=perturb_y
    )

    plot(ky_vals, evals', legend=false, color=:black, xlabel="k_y", ylabel="Energy")

    folder_name = joinpath("plots", "ribbon_spectrum")
    isdir(folder_name) || mkpath(folder_name)
    savefig(joinpath(folder_name, "ribbon_spectrum_A$(A)_B$(B)_m$(m)_gamma$(gamma)_perturbation_$(perturbation_type).png"))
end

### OBC LDOS Verification

In [ ]:
using LinearAlgebra
using Printf
using Plots
using DataFrames
using SparseArrays
using ProgressMeter
using Statistics
using LaTeXStrings
using JLD2: @save, @load
using Base.Threads

println("Julia Threads: ", Threads.nthreads())
println("BLAS Threads:  ", LinearAlgebra.BLAS.get_num_threads())


## Pauli matrices
sigma_x = [0 1; 1 0]
sigma_y = [0 -im; im 0]
sigma_z = [1 0; 0 -1]
identity = [1 0; 0 1]

# 1. Unified Block Generator
function real_space_perturbed_qwz_blocks(; 
    A::Real=1.0, 
    B::Real=1.0, 
    m::Real=0.0, 
    gamma::Real=0.0, 
    perturbation_type::Symbol=:none
)
    # Base QWZ blocks
    onsite = ComplexF64.((m + 2.0 * B) * sigma_z)
    tx = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_x)
    ty = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_y)

    # Apply perturbations
    if perturbation_type == :symmetric
        # gamma * (cos(kx) + cos(ky)) * I
        tx += ComplexF64.(0.5 * gamma * identity)
        ty += ComplexF64.(0.5 * gamma * identity)
    elseif perturbation_type == :tilt
        # gamma * sin(kx) * I
        tx += ComplexF64.(0.5im * gamma * identity)
        # ty is unperturbed by the 1D tilt
    elseif perturbation_type == :none
        # Base QWZ, do nothing
    else
        error("Unknown perturbation type: $perturbation_type")
    end

    return onsite, tx, ty
end

# Map a site and orbital to a matrix index in the 2LxLy basis.
site_index_qwz(x::Int, y::Int, orb::Int, Lx::Int, Ly::Int) = 2 * ((y - 1) * Lx + (x - 1)) + orb


# 2. Updated Real-Space Hamiltonian Builder
function real_space_perturbed_hamiltonian_qwz(
    Lx::Int,
    Ly::Int;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma::Real=0.0,
    perturbation_type::Symbol=:none,
    periodic_x::Bool=false,
    periodic_y::Bool=false,
    sparse_output::Bool=true
)
    Lx > 0 || error("Lx must be positive.")
    Ly > 0 || error("Ly must be positive.")

    # Get the perturbed blocks
    onsite, tx, ty = real_space_perturbed_qwz_blocks(; A=A, B=B, m=m, gamma=gamma, perturbation_type=perturbation_type)
    
    nsites = Lx * Ly
    dim = 2 * nsites
    H = sparse_output ? spzeros(ComplexF64, dim, dim) : zeros(ComplexF64, dim, dim)

    function add_block!(mat, row_site::Tuple{Int, Int}, col_site::Tuple{Int, Int}, block::AbstractMatrix{<:Number})
        (xr, yr) = row_site
        (xc, yc) = col_site
        row_base = site_index_qwz(xr, yr, 1, Lx, Ly)
        col_base = site_index_qwz(xc, yc, 1, Lx, Ly)
        @inbounds for a in 0:1, b in 0:1
            mat[row_base + a, col_base + b] += ComplexF64(block[a + 1, b + 1])
        end
        return nothing
    end

    for y in 1:Ly, x in 1:Lx
        add_block!(H, (x, y), (x, y), onsite)

        if x < Lx
            add_block!(H, (x + 1, y), (x, y), tx)
            add_block!(H, (x, y), (x + 1, y), tx')
        elseif periodic_x
            add_block!(H, (1, y), (x, y), tx)
            add_block!(H, (x, y), (1, y), tx')
        end

        if y < Ly
            add_block!(H, (x, y + 1), (x, y), ty)
            add_block!(H, (x, y), (x, y + 1), ty')
        elseif periodic_y
            add_block!(H, (x, 1), (x, y), ty)
            add_block!(H, (x, y), (x, 1), ty')
        end
    end

    return H
end

function k_space_perturbed_qwz_hamiltonian(
    kx::Real,
    ky::Real;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma::Real=0.0,
    perturbation_type::Symbol=:none
)::Matrix{ComplexF64}

    # Base QWZ Hamiltonian in k-space
    H_k = (m + 2B - B * (cos(kx) + cos(ky))) * sigma_z +
          A * sin(kx) * sigma_x +
          A * sin(ky) * sigma_y

    # Apply perturbations
    if perturbation_type == :symmetric
        H_k += gamma * (cos(kx) + cos(ky)) * identity
    elseif perturbation_type == :tilt
        H_k += gamma * sin(kx) * identity
    elseif perturbation_type == :none
        # Do nothing
    else
        error("Unknown perturbation type: $perturbation_type")
    end

    return ComplexF64.(H_k)
end

# 1. Total Density of States (DOS)
function compute_dos(
    eigs::Vector{Float64}, 
    E_range::AbstractVector{<:Real}; 
    eta=0.05
)
    dos = zeros(Float64, length(E_range))
    gaussian(E, E0, sigma) = (1 / (sigma * sqrt(2π))) * exp(-0.5 * ((E - E0) / sigma)^2)
    
    for (i, E_target) in enumerate(E_range)
        dos[i] = sum(gaussian.(eigs, E_target, eta))
    end
    return dos
end

# 2. Local Density of States (LDOS)
function compute_ldos(
    F::Eigen, 
    Lx::Int, 
    Ly::Int, 
    target_E::Real; 
    eta::Real=0.05
)
    ldos = zeros(Float64, Lx, Ly)
    eigs = real(F.values)
    
    # Gaussian broadening function
    gaussian(E, E0, sigma) = (1 / (sigma * sqrt(2π))) * exp(-0.5 * ((E - E0) / sigma)^2)
    
    for i in 1:length(eigs)
        weight = gaussian(eigs[i], target_E, eta)
        
        # Optimization: only sum states that are actually close to target_E
        if weight > 1e-5 
            psi = F.vectors[:, i]
            for y in 1:Ly, x in 1:Lx
                base = 2 * ((y - 1) * Lx + (x - 1))
                prob_density = abs2(psi[base + 1]) + abs2(psi[base + 2])
                ldos[x, y] += weight * prob_density
            end
        end
    end
    
    return ldos
end



# Example usage to plot:
A = 1.0
B = 1.0
m = -1.0
Lxs = [25]
Lys = [25]
gammas = [0.0, 0.5, 1.0, 2.0]
perturbation_type = :symmetric

periodic_x = false
periodic_y = false

# # OBC spectrum and min|E| eigenstate density
# @showprogress for (Lx, Ly) in zip(Lxs, Lys)
#     for gamma in gammas
#         H_obc = real_space_perturbed_hamiltonian_qwz(
#             Lx, Ly; 
#             A=A, 
#             B=B, 
#             m=m, 
#             gamma=gamma, 
#             perturbation_type=perturbation_type, 
#             periodic_x=periodic_x, 
#             periodic_y=periodic_y, 
#             sparse_output=false
#         )

#         F = eigen(H_obc)
#         eigs = real(F.values)
#         order = sortperm(abs.(eigs))
#         eigs_sorted = eigs[sortperm(eigs)]

#         edge_idx = order[1]
#         psi = F.vectors[:, edge_idx]
#         edge_density = zeros(Float64, Lx, Ly)
#         for y in 1:Ly, x in 1:Lx
#             base = 2 * ((y - 1) * Lx + (x - 1))
#             edge_density[x, y] = abs2(psi[base + 1]) + abs2(psi[base + 2])
#         end

#         p1 = scatter(
#             1:length(eigs_sorted), eigs_sorted;
#             label="OBC eigenvalues",
#             markerstrokewidth=0,
#             markersize=3,
#             color=:slateblue,
#             xlabel="sorted state index",
#             ylabel="energy",
#             title="Open-boundary spectrum"
#         )
#         hline!(p1, [0.0]; linestyle=:dash, color=:black, label=false)

#         p2 = heatmap(
#             1:Lx, 1:Ly, edge_density';
#             xlabel="x", ylabel="y",
#             title="Lowest-|E| eigenstate density",
#             color=:viridis,
#             aspect_ratio=1,
#             colorbar_title="|ψ|²"
#         )

#         plot(p1, p2; layout=(1, 2), size=(800, 600))

#         foldername = joinpath("plots", "pert_obc_spec", "m$(m)")
#         isdir(foldername) || mkpath(foldername)

#         savefig(joinpath(foldername, "qwz_obc_edge_state_A$(A)_B$(B)_m$(m)_gamma$(gamma)_perturbation_$(perturbation_type)_Lx$(Lx)_Ly$(Ly).png"))
#     end
# end


# # Calculate LDOS at E = 0.0 with a broadening of 0.1
# target_E = 2.0
# broadening_width = 0.1

# @showprogress for (Lx, Ly) in zip(Lxs, Lys)
#     for gamma in gammas
#         H_obc = real_space_perturbed_hamiltonian_qwz(
#             Lx, Ly; 
#             A=A, 
#             B=B, 
#             m=m, 
#             gamma=gamma, 
#             perturbation_type=perturbation_type, 
#             periodic_x=periodic_x, 
#             periodic_y=periodic_y, 
#             sparse_output=false
#         )

#         F = eigen(H_obc)

#         ldos_heatmap = compute_ldos(
#             F, 
#             Lx, 
#             Ly, 
#             target_E, 
#             eta=broadening_width
#         )

#         p3 = heatmap(
#             1:Lx, 1:Ly, ldos_heatmap';
#             xlabel="x", ylabel="y",
#             title="LDOS at E = $(target_E), eta = $(broadening_width)",
#             color=:viridis,
#             aspect_ratio=1,
#             colorbar_title="ρ(x,y,E)"
#         )

#         plot(p3; size=(400, 400))

#         foldername = joinpath("plots", "pert_obc_spec", "m$(m)")
#         isdir(foldername) || mkpath(foldername)

#         savefig(joinpath(foldername, "qwz_obc_ldos_A$(A)_B$(B)_m$(m)_gamma$(gamma)_perturbation_$(perturbation_type)_Lx$(Lx)_Ly$(Ly)_E$(target_E)_eta$(broadening_width).png"))
#     end
# end

## Disorder

In [ ]:
# 1. Unified Block Generator (For Clean/Global Terms)
function real_space_perturbed_qwz_blocks(; 
    A::Real=1.0, 
    B::Real=1.0, 
    m::Real=0.0, 
    gamma::Real=0.0, 
    perturbation_type::Symbol=:none
)
    # Base QWZ blocks
    onsite_base = ComplexF64.((m + 2.0 * B) * sigma_z)
    tx = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_x)
    ty = ComplexF64.(-0.5 * B * sigma_z - 0.5im * A * sigma_y)

    # Apply global momentum-derived perturbations
    if perturbation_type == :symmetric
        tx += ComplexF64.(0.5 * gamma * identity)
        ty += ComplexF64.(0.5 * gamma * identity)
    elseif perturbation_type == :tilt
        tx += ComplexF64.(0.5im * gamma * identity)
    elseif perturbation_type == :none
        # Do nothing
    else
        error("Unknown perturbation type: $perturbation_type")
    end

    return onsite_base, tx, ty
end

# Map a site and orbital to a matrix index in the 2LxLy basis.
site_index_qwz(x::Int, y::Int, orb::Int, Lx::Int, Ly::Int) = 2 * ((y - 1) * Lx + (x - 1)) + orb

# 2. Updated Real-Space Hamiltonian Builder (With Disorder)
function real_space_perturbed_disordered_hamiltonian_qwz(
    Lx::Int,
    Ly::Int;
    A::Real=1.0,
    B::Real=1.0,
    m::Real=0.0,
    gamma::Real=0.0,
    perturbation_type::Symbol=:none,
    disorder_type::Symbol=:none,
    W::Real=0.0, # Disorder strength
    periodic_x::Bool=false,
    periodic_y::Bool=false,
    sparse_output::Bool=true
)
    Lx > 0 || error("Lx must be positive.")
    Ly > 0 || error("Ly must be positive.")

    # Get the base blocks (without site-specific disorder)
    onsite_base, tx, ty = real_space_perturbed_qwz_blocks(; A=A, B=B, m=m, gamma=gamma, perturbation_type=perturbation_type)
    
    nsites = Lx * Ly
    dim = 2 * nsites
    H = sparse_output ? spzeros(ComplexF64, dim, dim) : zeros(ComplexF64, dim, dim)

    function add_block!(mat, row_site::Tuple{Int, Int}, col_site::Tuple{Int, Int}, block::AbstractMatrix{<:Number})
        (xr, yr) = row_site
        (xc, yc) = col_site
        row_base = site_index_qwz(xr, yr, 1, Lx, Ly)
        col_base = site_index_qwz(xc, yc, 1, Lx, Ly)
        @inbounds for a in 0:1, b in 0:1
            mat[row_base + a, col_base + b] += ComplexF64(block[a + 1, b + 1])
        end
        return nothing
    end

    for y in 1:Ly, x in 1:Lx
        
        # --- APPLY SITE-SPECIFIC DISORDER ---
        onsite_xy = copy(onsite_base)
        
        if W > 0.0
            # Generate a random scalar in the range [-W/2, W/2]
            random_val = W * (rand() - 0.5)
            
            if disorder_type == :anderson
                # Chemical potential disorder (shifts both orbitals equally)
                onsite_xy += random_val * identity
            elseif disorder_type == :mass
                # Mass/gap disorder (shifts orbitals oppositely)
                onsite_xy += random_val * sigma_z
            elseif disorder_type != :none
                error("Unknown disorder type: $disorder_type")
            end
        end

        # Add the disordered on-site block
        add_block!(H, (x, y), (x, y), onsite_xy)

        # Add the clean hopping blocks
        if x < Lx
            add_block!(H, (x + 1, y), (x, y), tx)
            add_block!(H, (x, y), (x + 1, y), tx')
        elseif periodic_x
            add_block!(H, (1, y), (x, y), tx)
            add_block!(H, (x, y), (1, y), tx')
        end

        if y < Ly
            add_block!(H, (x, y + 1), (x, y), ty)
            add_block!(H, (x, y), (x, y + 1), ty')
        elseif periodic_y
            add_block!(H, (x, 1), (x, y), ty)
            add_block!(H, (x, y), (x, 1), ty')
        end
    end

    return H
end

In [ ]:
## plotting disordered OBC stuff
A = 1.0
B = 1.0
m = -1.0
Lxs = [50]
Lys = [50]
gammas = [1.1] #[0.0, 0.5, 1.0, 2.0]
perturbation_type = :symmetric
disorder_type = :anderson
Ws = [0.0, 0.01, 0.05, 0.1, 0.25] #[0.0, 0.5, 1.0, 2.0]

# Calculate LDOS at E = 0.0 with a broadening of 0.1
target_E = 0.0
broadening_width = 0.1

periodic_x = false
periodic_y = false

# OBC spectrum and min|E| eigenstate density
@showprogress for (Lx, Ly) in zip(Lxs, Lys)
    for gamma in gammas, W in Ws
        H_obc = real_space_perturbed_disordered_hamiltonian_qwz(
            Lx, Ly; 
            A=A, 
            B=B, 
            m=m, 
            gamma=gamma, 
            perturbation_type=perturbation_type, 
            periodic_x=periodic_x, 
            periodic_y=periodic_y, 
            disorder_type=disorder_type,
            W=W,
            sparse_output=false
        )

        F = eigen(H_obc)
        eigs = real(F.values)
        order = sortperm(abs.(eigs))
        eigs_sorted = eigs[sortperm(eigs)]

        edge_idx = order[1]
        psi = F.vectors[:, edge_idx]
        edge_density = zeros(Float64, Lx, Ly)
        for y in 1:Ly, x in 1:Lx
            base = 2 * ((y - 1) * Lx + (x - 1))
            edge_density[x, y] = abs2(psi[base + 1]) + abs2(psi[base + 2])
        end

        p1 = scatter(
            1:length(eigs_sorted), eigs_sorted;
            label="OBC eigenvalues",
            markerstrokewidth=0,
            markersize=3,
            color=:slateblue,
            xlabel="sorted state index",
            ylabel="energy",
            title="Open-boundary spectrum"
        )
        hline!(p1, [0.0]; linestyle=:dash, color=:black, label=false)

        p2 = heatmap(
            1:Lx, 1:Ly, edge_density';
            xlabel="x", ylabel="y",
            title="Lowest-|E| eigenstate density",
            color=:viridis,
            aspect_ratio=1,
            colorbar_title="|ψ|²"
        )

        plot(p1, p2; layout=(1, 2), size=(800, 600))

        foldername = joinpath("plots", "pert_disordered_obc_spec", "m$(m)")
        isdir(foldername) || mkpath(foldername)

        savefig(joinpath(foldername, "qwz_obc_edge_state_A$(A)_B$(B)_m$(m)_gamma$(gamma)_perturbation_$(perturbation_type)_Lx$(Lx)_Ly$(Ly)_disorder_$(disorder_type)_W$(W).png"))

                ldos_heatmap = compute_ldos(
            F, 
            Lx, 
            Ly, 
            target_E, 
            eta=broadening_width
        )

        p3 = heatmap(
            1:Lx, 1:Ly, ldos_heatmap';
            xlabel="x", ylabel="y",
            title="LDOS at E = $(target_E), eta = $(broadening_width)",
            color=:viridis,
            aspect_ratio=1,
            colorbar_title="ρ(x,y,E)"
        )

        plot(p3; size=(400, 400))

        foldername = joinpath("plots", "pert_disordered_obc_spec", "m$(m)")
        isdir(foldername) || mkpath(foldername)

        savefig(joinpath(foldername, "qwz_obc_ldos_A$(A)_B$(B)_m$(m)_gamma$(gamma)_perturbation_$(perturbation_type)_Lx$(Lx)_Ly$(Ly)_disorder_$(disorder_type)_W$(W)_E$(target_E)_eta$(broadening_width).png"))
    
    end
end


# @showprogress for (Lx, Ly) in zip(Lxs, Lys)
#     for gamma in gammas
#         H_obc = real_space_perturbed_disordered_hamiltonian_qwz(
#             Lx, Ly; 
#             A=A, 
#             B=B, 
#             m=m, 
#             gamma=gamma, 
#             perturbation_type=perturbation_type, 
#             periodic_x=periodic_x, 
#             periodic_y=periodic_y, 
#             disorder_type=disorder_type,
#             W=W,
#             sparse_output=false
#         )

#         F = eigen(H_obc)

#         ldos_heatmap = compute_ldos(
#             F, 
#             Lx, 
#             Ly, 
#             target_E, 
#             eta=broadening_width
#         )

#         p3 = heatmap(
#             1:Lx, 1:Ly, ldos_heatmap';
#             xlabel="x", ylabel="y",
#             title="LDOS at E = $(target_E), eta = $(broadening_width)",
#             color=:viridis,
#             aspect_ratio=1,
#             colorbar_title="ρ(x,y,E)"
#         )

#         plot(p3; size=(400, 400))

#         foldername = joinpath("plots", "pert_obc_spec", "m$(m)")
#         isdir(foldername) || mkpath(foldername)

#         savefig(joinpath(foldername, "qwz_obc_ldos_A$(A)_B$(B)_m$(m)_gamma$(gamma)_perturbation_$(perturbation_type)_Lx$(Lx)_Ly$(Ly)_E$(target_E)_eta$(broadening_width).png"))
#     end
# end